In [4]:
from pathlib import Path
import pandas as pd

file_path = (
    Path("..")
    / "data"
    / "raw"
    / "performance"
    / "2019-09-18-subway-on-time-performance-v1.parquet"
)

df = pd.read_parquet(file_path)

df.head()

,stop_sequence,stop_id,parent_station,move_timestamp,stop_timestamp,travel_time_seconds,dwell_time_seconds,headway_trunk_seconds,headway_branch_seconds,service_date,...,trip_id,vehicle_label,vehicle_consist,direction,direction_destination,scheduled_arrival_time,scheduled_departure_time,scheduled_travel_time,scheduled_headway_branch,scheduled_headway_trunk
0,310,70107,place-lake,NaN,1.568794e+09,NaN,NaN,NaN,NaN,20190918,...,ADDED-1568746059,3870,3870,West,Boston College,23100.0,23100.0,180.0,NaN,NaN
1,310,70107,place-lake,NaN,1.568794e+09,NaN,NaN,NaN,NaN,20190918,...,ADDED-1568746058,3692,3692,West,Boston College,23100.0,23100.0,180.0,NaN,NaN
2,1,Alewife-02,place-alfcl,NaN,1.568794e+09,NaN,NaN,NaN,NaN,20190918,...,ADDED-1568746060,1872,1872|1873|1810|1811|1820|1821,South,Ashmont/Braintree,18960.0,18960.0,NaN,NaN,NaN
3,1,Alewife-01,place-alfcl,NaN,1.568794e+09,NaN,NaN,459.0,NaN,20190918,...,ADDED-1568746061,1867,1867|1866|1814|1815|1806|1807,South,Ashmont/Braintree,19440.0,19440.0,NaN,NaN,480.0
4,1,Braintree-01,place-brntn,NaN,1.568794e+09,NaN,NaN,NaN,NaN,20190918,...,ADDED-1568746062,1856,1856|1857|1879|1878|1819|1818,North,Alewife,18780.0,18780.0,NaN,NaN,NaN


### Initial Dataset Inspection

The MBTA Subway On-Time Performance dataset for September 18, 2019 was successfully loaded from the raw Parquet file into a pandas DataFrame.

The initial preview confirms that the dataset contains operational records at the stop level, with fields covering:

- Stop and station information
- Operational timestamps
- Travel and dwell times
- Trunk and branch headways
- Route and trip identifiers
- Vehicle information
- Direction and destination
- Scheduled arrival, departure, travel time, and headway information

The first five records also show that missing values are present in several operational and headway-related fields. At this stage, these missing values are only being observed; no cleaning, removal, or imputation decisions have been made.

In [5]:
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Rows: 44,970
Columns: 27


### Dataset Dimensions

The dataset contains **44,970 stop-level records across 27 columns**.

This provides the initial scope of the analytical dataset. The relatively large number of records allows subsequent profiling to examine missingness and operational patterns across routes, stops, trips, vehicles, and timestamps.

No records have been removed or transformed at this stage.

In [6]:
df.columns.tolist()

['stop_sequence',
 'stop_id',
 'parent_station',
 'move_timestamp',
 'stop_timestamp',
 'travel_time_seconds',
 'dwell_time_seconds',
 'headway_trunk_seconds',
 'headway_branch_seconds',
 'service_date',
 'route_id',
 'direction_id',
 'start_time',
 'vehicle_id',
 'branch_route_id',
 'trunk_route_id',
 'stop_count',
 'trip_id',
 'vehicle_label',
 'vehicle_consist',
 'direction',
 'direction_destination',
 'scheduled_arrival_time',
 'scheduled_departure_time',
 'scheduled_travel_time',
 'scheduled_headway_branch',
 'scheduled_headway_trunk']

### Column Overview

The dataset contains 27 columns covering several aspects of subway operations:

- **Stop and station information:** `stop_sequence`, `stop_id`, `parent_station`
- **Operational timing:** `move_timestamp`, `stop_timestamp`, `travel_time_seconds`, `dwell_time_seconds`
- **Headway measurements:** `headway_trunk_seconds`, `headway_branch_seconds`
- **Service and route information:** `service_date`, `route_id`, `direction_id`, `start_time`, `branch_route_id`, `trunk_route_id`, `stop_count`
- **Trip and vehicle information:** `trip_id`, `vehicle_id`, `vehicle_label`, `vehicle_consist`
- **Direction and destination:** `direction`, `direction_destination`
- **Scheduled service information:** `scheduled_arrival_time`, `scheduled_departure_time`, `scheduled_travel_time`, `scheduled_headway_branch`, `scheduled_headway_trunk`

This column structure provides both observed operational measurements and corresponding scheduled service information, which will allow later analysis to compare actual and scheduled performance.

At this stage, the columns are being documented based on their names and observed structure. Their exact relationships and semantics will be investigated during subsequent profiling.

In [7]:
df.iloc[0]

stop_sequence                            310
stop_id                                70107
parent_station                    place-lake
move_timestamp                           NaN
stop_timestamp                  1568793600.0
travel_time_seconds                      NaN
dwell_time_seconds                       NaN
headway_trunk_seconds                    NaN
headway_branch_seconds                   NaN
service_date                        20190918
route_id                             Green-B
direction_id                           False
start_time                             14400
vehicle_id                           G-10005
branch_route_id                      Green-B
trunk_route_id                         Green
stop_count                                 1
trip_id                     ADDED-1568746059
vehicle_label                           3870
vehicle_consist                         3870
direction                               West
direction_destination         Boston College
scheduled_

### First Record Inspection

The first individual record was inspected to examine the values and missing
fields at the row level.

This record belongs to the `Green-B` route and contains identifiers,
vehicle information, direction, destination, and scheduled timing
information. Several observed operational fields are missing in this
record, including `move_timestamp`, `travel_time_seconds`,
`dwell_time_seconds`, both observed headway fields, and both scheduled
headway fields.

The record also contains an `ADDED-*` trip identifier and a populated
scheduled travel time, illustrating that missing operational measurements
can coexist with otherwise populated trip, vehicle, route, and schedule
information.

This inspection provides an example of the row-level missingness observed
in the initial dataset preview. It does not by itself establish the reason
for the missing values; subsequent profiling examines the missingness
patterns across the full dataset.

In [8]:
df.dtypes

stop_sequence                 int16
stop_id                         str
parent_station                  str
move_timestamp              float64
stop_timestamp              float64
travel_time_seconds         float64
dwell_time_seconds          float64
headway_trunk_seconds       float64
headway_branch_seconds      float64
service_date                  int64
route_id                        str
direction_id                   bool
start_time                    int64
vehicle_id                      str
branch_route_id                 str
trunk_route_id                  str
stop_count                    int16
trip_id                         str
vehicle_label                   str
vehicle_consist                 str
direction                       str
direction_destination           str
scheduled_arrival_time      float64
scheduled_departure_time    float64
scheduled_travel_time       float64
scheduled_headway_branch    float64
scheduled_headway_trunk     float64
dtype: object

### Data Types

The dataset contains a mixture of numeric, boolean, and string-like fields.

- `stop_sequence` and `stop_count` are stored as `int16`.
- `service_date` and `start_time` are stored as `int64`.
- Operational timing, headway, and scheduled timing fields are stored as `float64`.
- `direction_id` is stored as a boolean.
- Route, stop, trip, vehicle, and descriptive fields are stored as string-like values.

The timing-related fields use `float64`, which allows them to represent missing values (`NaN`) alongside numeric observations.

At this stage, the observed dtypes are being recorded rather than changed. Any type conversions will be considered later based on the analytical requirements and the semantics of the fields.

In [9]:
df.isna().sum().sort_values(ascending=False)

headway_branch_seconds      14588
scheduled_headway_branch    12451
branch_route_id             11961
dwell_time_seconds           3818
headway_trunk_seconds        3603
scheduled_travel_time        2258
travel_time_seconds          1680
move_timestamp               1549
scheduled_headway_trunk       552
scheduled_departure_time      217
scheduled_arrival_time        217
stop_timestamp                126
vehicle_label                  11
route_id                        0
parent_station                  0
stop_sequence                   0
service_date                    0
stop_id                         0
start_time                      0
trip_id                         0
stop_count                      0
trunk_route_id                  0
vehicle_id                      0
direction_id                    0
direction_destination           0
vehicle_consist                 0
direction                       0
dtype: int64

### Missing Value Profile

An initial missing-value assessment was performed across all 27 columns.

The largest numbers of missing observations occur in:

- `headway_branch_seconds`: 14,588 missing values (32.44%)
- `scheduled_headway_branch`: 12,451 (27.69%)
- `branch_route_id`: 11,961 (26.60%)
- `dwell_time_seconds`: 3,818 (8.49%)
- `headway_trunk_seconds`: 3,603 (8.01%)
- `scheduled_travel_time`: 2,258 (5.02%)
- `travel_time_seconds`: 1,680 (3.74%)
- `move_timestamp`: 1,549 (3.44%)

The remaining fields have substantially lower missingness, with
`vehicle_label` having only 11 missing observations (0.02%).

Several identifier and descriptive fields have no missing values,
including `route_id`, `stop_id`, `trip_id`, `vehicle_id`, `parent_station`,
`direction`, and `direction_destination`.

This profile shows that missingness is concentrated in specific operational,
headway, and timing fields rather than being uniformly distributed across
the dataset.

At this stage, the missing values are only being profiled. No rows have
been dropped and no values have been imputed or replaced. Subsequent
analysis will investigate whether the missingness is structural,
operational, or indicative of data-quality issues.

In [10]:
missing = df.isna().sum()

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": (missing / len(df) * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary["missing_count"] > 0
].sort_values("missing_count", ascending=False)

missing_summary

,missing_count,missing_percent
headway_branch_seconds,14588,32.44
scheduled_headway_branch,12451,27.69
branch_route_id,11961,26.60
dwell_time_seconds,3818,8.49
headway_trunk_seconds,3603,8.01
scheduled_travel_time,2258,5.02
travel_time_seconds,1680,3.74
move_timestamp,1549,3.44
scheduled_headway_trunk,552,1.23
scheduled_departure_time,217,0.48


### Missing Value Summary

The missing-value counts were converted into percentages of the full
44,970-row dataset to provide a comparable measure of missingness across
columns.

The highest missingness occurs in the branch-level headway and route fields:

- `headway_branch_seconds`: 14,588 missing (32.44%)
- `scheduled_headway_branch`: 12,451 missing (27.69%)
- `branch_route_id`: 11,961 missing (26.60%)

Other operational fields have lower but still notable missingness,
including `dwell_time_seconds` (8.49%), `headway_trunk_seconds` (8.01%),
`scheduled_travel_time` (5.02%), `travel_time_seconds` (3.74%), and
`move_timestamp` (3.44%).

The remaining fields have relatively low missingness, with
`scheduled_arrival_time` and `scheduled_departure_time` each at 0.48%,
`stop_timestamp` at 0.28%, and `vehicle_label` at 0.02%.

This confirms that missingness is concentrated in particular operational
and branch-related fields rather than being evenly distributed across the
dataset.

These percentages describe the extent of missingness only. They do not yet
establish why the values are missing or whether the missingness is
structural, operational, or a data-quality issue. Those questions are
investigated in the following profiling steps.

In [11]:
branch_missing = df[df["branch_route_id"].isna()]

branch_missing["route_id"].value_counts()

route_id
Orange      6019
Blue        4327
Mattapan    1605
Red           10
Name: count, dtype: int64

### `branch_route_id` Missingness by Route

The records where `branch_route_id` is missing were grouped by `route_id` to
identify whether the missingness is distributed evenly across the subway
routes.

The 11,961 missing `branch_route_id` values are concentrated in four
routes:

- Orange: 6,019 records
- Blue: 4,327 records
- Mattapan: 1,605 records
- Red: 10 records

No Green branch records appear in this initial missing-value grouping.

This indicates that `branch_route_id` missingness is strongly associated
with route type rather than being uniformly distributed across the dataset.

The route-level percentages are examined next to determine whether these
counts represent systematic or structural missingness for particular
routes.

In [12]:
branch_route_profile = (
    df.groupby("route_id")["branch_route_id"]
    .agg(
        total_rows="size",
        missing_rows=lambda s: s.isna().sum(),
        non_missing_rows=lambda s: s.notna().sum()
    )
)

branch_route_profile["missing_percent"] = (
    branch_route_profile["missing_rows"]
    / branch_route_profile["total_rows"]
    * 100
).round(2)

branch_route_profile.sort_values("missing_percent", ascending=False)

,total_rows,missing_rows,non_missing_rows,missing_percent
route_id,,,,
Blue,4327,4327,0,100.00
Mattapan,1605,1605,0,100.00
Orange,6019,6019,0,100.00
Red,7600,10,7590,0.13
Green-D,5465,0,5465,0.00
Green-C,6293,0,6293,0.00
Green-B,7611,0,7611,0.00
Green-E,6050,0,6050,0.00


### `branch_route_id` Missingness by Route

The route-level missingness profile shows that `branch_route_id` is
systematically associated with specific routes.

- `Blue`: 4,327 of 4,327 records missing (100.00%)
- `Mattapan`: 1,605 of 1,605 records missing (100.00%)
- `Orange`: 6,019 of 6,019 records missing (100.00%)
- `Red`: 10 of 7,600 records missing (0.13%)
- `Green-D`: 0 of 5,465 records missing (0.00%)
- `Green-C`: 0 of 6,293 records missing (0.00%)
- `Green-B`: 0 of 7,611 records missing (0.00%)
- `Green-E`: 0 of 6,050 records missing (0.00%)

The missingness is therefore strongly route-structured rather than random.
Blue, Mattapan, and Orange have no non-missing `branch_route_id` values,
while all Green branches have complete values. Red has only 10 missing
records out of 7,600.

This pattern suggests that `branch_route_id` is likely structurally
inapplicable for Blue, Mattapan, and Orange rather than representing
randomly corrupted records.

No values are filled or removed at this stage. The field will be retained
as missing for these records until its role and applicability are confirmed
during later data-quality and transformation work.

In [13]:
headway_missing_profile = (
    df.groupby("route_id")
    .agg(
        total_rows=("route_id", "size"),
        branch_headway_missing=(
            "headway_branch_seconds",
            lambda s: s.isna().sum()
        ),
        scheduled_branch_headway_missing=(
            "scheduled_headway_branch",
            lambda s: s.isna().sum()
        )
    )
)

headway_missing_profile["branch_headway_missing_percent"] = (
    headway_missing_profile["branch_headway_missing"]
    / headway_missing_profile["total_rows"] * 100
).round(2)

headway_missing_profile["scheduled_branch_headway_missing_percent"] = (
    headway_missing_profile["scheduled_branch_headway_missing"]
    / headway_missing_profile["total_rows"] * 100
).round(2)

headway_missing_profile.sort_values(
    "branch_headway_missing_percent",
    ascending=False
)

,total_rows,branch_headway_missing,scheduled_branch_headway_missing,branch_headway_missing_percent,scheduled_branch_headway_missing_percent
route_id,,,,,
Blue,4327,4327,4327,100.00,100.00
Mattapan,1605,1605,1605,100.00,100.00
Orange,6019,6019,6019,100.00,100.00
Red,7600,773,153,10.17,2.01
Green-D,5465,508,81,9.30,1.48
Green-C,6293,483,96,7.68,1.53
Green-E,6050,408,12,6.74,0.20
Green-B,7611,465,158,6.11,2.08


### Branch Headway Missingness by Route

Branch headway missingness was examined by route for both the observed
`headway_branch_seconds` field and the corresponding
`scheduled_headway_branch` field.

The results show a clear distinction between routes:

- **Blue:** 100.00% missing for both branch headway fields
- **Mattapan:** 100.00% missing for both branch headway fields
- **Orange:** 100.00% missing for both branch headway fields
- **Red:** 10.17% missing observed branch headway and 2.01% missing scheduled branch headway
- **Green-D:** 9.30% and 1.48%, respectively
- **Green-C:** 7.68% and 1.53%, respectively
- **Green-E:** 6.74% and 0.20%, respectively
- **Green-B:** 6.11% and 2.08%, respectively

The complete absence of both branch headway measures for Blue, Mattapan,
and Orange aligns with the earlier `branch_route_id` pattern. This suggests
that branch headway fields are likely structurally inapplicable to these
routes rather than randomly missing.

For the Red and Green routes, branch headway missingness is partial rather
than complete, indicating a different missingness pattern that requires
further investigation.

No values are imputed or removed at this stage.

In [14]:
timestamp_missing_profile = (
    df[["move_timestamp", "stop_timestamp"]]
    .isna()
    .value_counts()
    .rename("row_count")
    .reset_index()
)

timestamp_missing_profile

,move_timestamp,stop_timestamp,row_count
0,False,False,43295
1,True,False,1549
2,False,True,126


### Timestamp Missingness Patterns

The missingness of `move_timestamp` and `stop_timestamp` was examined jointly
to determine whether the two operational timestamps tend to be missing
together or independently.

The dataset contains three observed missingness patterns:

- **43,295 records (96.28%)** have both `move_timestamp` and `stop_timestamp` present.
- **1,549 records (3.44%)** have `move_timestamp` missing while `stop_timestamp` is present.
- **126 records (0.28%)** have `stop_timestamp` missing while `move_timestamp` is present.

There are no records in which both timestamps are missing.

The missingness is therefore predominantly associated with one timestamp at a
time rather than simultaneous loss of both timestamps. In particular,
missing `move_timestamp` is substantially more common than missing
`stop_timestamp`.

This pattern is important for interpreting downstream travel-time
calculations because a travel-time value depends on the availability of both
timestamps. The relationship between timestamp availability and
`travel_time_seconds` is investigated next.

In [15]:
travel_missing_profile = (
    df.groupby(
        [
            df["travel_time_seconds"].isna().rename("travel_time_missing"),
            df["move_timestamp"].isna().rename("move_timestamp_missing"),
            df["stop_timestamp"].isna().rename("stop_timestamp_missing"),
        ]
    )
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

travel_missing_profile

,travel_time_missing,move_timestamp_missing,stop_timestamp_missing,row_count
0,False,False,False,43290
3,True,True,False,1549
2,True,False,True,126
1,True,False,False,5


### Travel Time Missingness Patterns

The missingness of `travel_time_seconds` was examined jointly with
`move_timestamp` and `stop_timestamp` to determine how travel-time
missingness relates to timestamp availability.

Four distinct patterns are present:

- **43,290 records** have all three fields present.
- **1,549 records** have `travel_time_seconds` and `move_timestamp` missing,
  while `stop_timestamp` is present.
- **126 records** have `travel_time_seconds` and `stop_timestamp` missing,
  while `move_timestamp` is present.
- **5 records** have `travel_time_seconds` missing while both timestamps are
  present.

Therefore, 1,675 of the 1,680 missing `travel_time_seconds` values occur
alongside a missing operational timestamp, while the remaining 5 records
have both timestamps available.

This shows that travel-time missingness is strongly associated with
timestamp availability. The five records where both timestamps are
present require separate investigation to determine why
`travel_time_seconds` is missing.

In [16]:
travel_missing_with_timestamps = df[
    df["travel_time_seconds"].isna()
    & df["move_timestamp"].notna()
    & df["stop_timestamp"].notna()
]

travel_missing_with_timestamps

,stop_sequence,stop_id,parent_station,move_timestamp,stop_timestamp,travel_time_seconds,dwell_time_seconds,headway_trunk_seconds,headway_branch_seconds,service_date,...,trip_id,vehicle_label,vehicle_consist,direction,direction_destination,scheduled_arrival_time,scheduled_departure_time,scheduled_travel_time,scheduled_headway_branch,scheduled_headway_trunk
5877,440,70237,place-clmnl,1.568808e+09,1.568808e+09,NaN,NaN,NaN,NaN,20190918,...,39988633,3822,3822,West,Cleveland Circle,33240.0,33240.0,60.0,360.0,360.0
20711,570,70161,place-river,1.568829e+09,1.568829e+09,NaN,NaN,NaN,NaN,20190918,...,39989564,3622-3849,3622|3849,West,Riverside,53580.0,53580.0,60.0,420.0,420.0
23810,170,70093,place-asmnl,1.568834e+09,1.568834e+09,NaN,NaN,NaN,NaN,20190918,...,ADDED-1568817507,1615,1615|1614|1517|1516|1724|1725,South,Ashmont/Braintree,57180.0,57180.0,120.0,600.0,600.0
34765,170,70093,place-asmnl,1.568849e+09,1.568849e+09,NaN,NaN,NaN,NaN,20190918,...,ADDED-1568817722,1701,1701|1700|1610|1611|1621|1620,South,Ashmont/Braintree,72120.0,72120.0,120.0,600.0,600.0
36791,180,70097,place-nqncy,1.568852e+09,1.568852e+09,NaN,874.0,1486.0,1486.0,20190918,...,41527514,1859,1859|1858|1827|1826|1831|1830,South,Ashmont/Braintree,76920.0,76920.0,480.0,660.0,660.0


### Finding: Travel-Time Missingness with Available Timestamps

Five records have missing `travel_time_seconds` values even though both `move_timestamp` and `stop_timestamp` are present.

This is a small but important exception: the missing travel time cannot be attributed simply to missing timestamps. These records therefore require further investigation to determine whether travel time can be reconstructed from the available timestamps and whether the missingness reflects a data-generation or processing issue.

The five records span the Green, Red, and related service data and include both normal and `ADDED-*` trip identifiers.

In [17]:
travel_missing_with_timestamps[
    [
        "move_timestamp",
        "stop_timestamp",
        "travel_time_seconds",
        "scheduled_travel_time",
        "route_id",
        "stop_id",
        "trip_id",
    ]
].assign(
    timestamp_difference=lambda x:
        x["stop_timestamp"] - x["move_timestamp"]
)

,move_timestamp,stop_timestamp,travel_time_seconds,scheduled_travel_time,route_id,stop_id,trip_id,timestamp_difference
5877,1.568808e+09,1.568808e+09,NaN,60.0,Green-C,70237,39988633,-6.0
20711,1.568829e+09,1.568829e+09,NaN,60.0,Green-D,70161,39989564,-45.0
23810,1.568834e+09,1.568834e+09,NaN,120.0,Red,70093,ADDED-1568817507,-76.0
34765,1.568849e+09,1.568849e+09,NaN,120.0,Red,70093,ADDED-1568817722,-33.0
36791,1.568852e+09,1.568852e+09,NaN,480.0,Red,70097,41527514,-431.0


### Finding: Timestamp Differences Are Invalid for These Records

Calculating `stop_timestamp - move_timestamp` for the five travel-time-missing records produces negative values for every record:

- Green-C: **-6 seconds**
- Green-D: **-45 seconds**
- Red: **-76 seconds**
- Red: **-33 seconds**
- Red: **-431 seconds**

Because the stop timestamp occurs before the move timestamp in all five cases, the timestamps cannot be used directly to reconstruct a valid travel time.

This indicates that the missing `travel_time_seconds` values are associated with **invalid timestamp ordering**, rather than simply being missing values that can be recovered through timestamp subtraction. The five records should therefore be treated as **exception cases** in the subsequent analysis.

In [18]:
exception_context = df.merge(
    travel_missing_with_timestamps[["trip_id", "stop_sequence"]],
    on="trip_id",
    suffixes=("", "_exception")
)

exception_context = exception_context[
    (
        exception_context["stop_sequence"]
        - exception_context["stop_sequence_exception"]
    ).abs() <= 20
]

exception_context[
    [
        "trip_id",
        "stop_sequence",
        "stop_id",
        "move_timestamp",
        "stop_timestamp",
        "travel_time_seconds",
        "dwell_time_seconds",
        "scheduled_travel_time",
    ]
].sort_values(
    ["trip_id", "stop_sequence"]
)

,trip_id,stop_sequence,stop_id,move_timestamp,stop_timestamp,travel_time_seconds,dwell_time_seconds,scheduled_travel_time
0,39988633,440,70237,1.568808e+09,1.568808e+09,NaN,NaN,60.0
1,39989564,570,70161,1.568829e+09,1.568829e+09,NaN,NaN,60.0
18,41527514,180,70097,1.568852e+09,1.568852e+09,NaN,874.0,480.0
17,41527514,190,70099,1.568852e+09,1.568853e+09,746.0,NaN,120.0
19,41527514,200,70101,1.568853e+09,1.568853e+09,124.0,119.0,180.0
2,ADDED-1568817507,170,70093,1.568834e+09,1.568834e+09,NaN,NaN,120.0
3,ADDED-1568817722,170,70093,1.568849e+09,1.568849e+09,NaN,NaN,120.0


### Finding: Exception Context Reveals Nearby Valid Trip Records

To understand the five exceptions in context, the missing records were matched back to their trips and nearby stop sequences within ±20 stops.

This provides an important comparison for the exceptions. In particular, trip `41527514` contains consecutive records at stop sequences **180, 190, and 200**. The travel time is missing at sequence 180, while the following segments have observed travel times of **746 seconds** and **124 seconds**, respectively.

The surrounding records therefore confirm that these are not isolated trips with no available trip context. Instead, at least some missing travel-time records occur within otherwise populated trip sequences, making them useful candidates for further investigation of how the underlying timestamps and travel-time fields were generated.

The other exceptions occur on separate trips, including the two `ADDED-*` trips, so they should be considered individually rather than assuming a single cause for all five records.

In [19]:
df[df["trip_id"] == "41527514"][
    [
        "stop_sequence",
        "stop_id",
        "move_timestamp",
        "stop_timestamp",
        "travel_time_seconds",
        "dwell_time_seconds",
        "scheduled_travel_time",
    ]
].sort_values("stop_sequence")

,stop_sequence,stop_id,move_timestamp,stop_timestamp,travel_time_seconds,dwell_time_seconds,scheduled_travel_time
34953,1,Alewife-01,NaN,1.568849e+09,NaN,602.0,NaN
35320,10,70063,1.568850e+09,1.568850e+09,120.0,56.0,120.0
35429,20,70065,1.568850e+09,1.568850e+09,71.0,51.0,120.0
35500,30,70067,1.568850e+09,1.568850e+09,101.0,59.0,180.0
35597,40,70069,1.568850e+09,1.568850e+09,176.0,51.0,240.0
35746,50,70071,1.568851e+09,1.568851e+09,82.0,52.0,180.0
35832,60,70073,1.568851e+09,1.568851e+09,65.0,50.0,120.0
35900,70,70075,1.568851e+09,1.568851e+09,63.0,100.0,180.0
36006,80,70077,1.568851e+09,1.568851e+09,17.0,59.0,120.0
36053,90,70079,1.568851e+09,1.568851e+09,24.0,16.0,60.0


### Finding: The Missing Travel Time Occurs at a Large Dwell-Time Boundary

Inspecting trip `41527514` across its stop sequence shows that the missing travel-time record occurs at **stop sequence 180**.

At this record:

- `travel_time_seconds` is missing.
- `move_timestamp` and `stop_timestamp` are present but incorrectly ordered.
- `dwell_time_seconds` is **874 seconds**.
- `scheduled_travel_time` is **480 seconds**.

The following stop at sequence 190 has a valid `travel_time_seconds` of **746 seconds**, followed by another valid value of **124 seconds** at sequence 200.

This provides stronger evidence that the missing value is associated with an anomalous record at the boundary of a very large dwell period, rather than representing an ordinary missing travel-time observation. The record should therefore remain classified as an exception rather than being imputed from the timestamp difference.

In [20]:
travel_relationship = df.sort_values(
    ["trip_id", "stop_sequence"]
).copy()

travel_relationship["previous_stop_timestamp"] = (
    travel_relationship.groupby("trip_id")["stop_timestamp"].shift(1)
)

travel_relationship["calculated_travel_time"] = (
    travel_relationship["move_timestamp"]
    - travel_relationship["previous_stop_timestamp"]
)

travel_relationship[
    ["travel_time_seconds", "calculated_travel_time"]
].dropna().describe()

,travel_time_seconds,calculated_travel_time
count,41550.000000,41550.000000
mean,80.576366,63.333959
std,69.751182,91.542424
min,2.000000,-1586.000000
25%,42.000000,38.250000
50%,69.000000,50.000000
75%,101.000000,65.000000
max,3405.000000,4971.000000


### Finding: Reconstructing Travel Time from Previous Stop Timestamps

To test whether travel time can be derived independently from the available timestamps, a `calculated_travel_time` was created using the current stop's `move_timestamp` minus the previous stop's `stop_timestamp` within each trip.

For the 41,550 records where both the observed and calculated travel times are available, the distributions differ substantially:

- Observed `travel_time_seconds`: mean **80.58 s**, median **69 s**
- Calculated travel time: mean **63.33 s**, median **50 s**
- The calculated measure also contains negative values, with a minimum of **-1,586 s**

Therefore, the timestamp-derived value does not reproduce the observed `travel_time_seconds` consistently. This suggests that the two fields are not interchangeable and that travel time should not be reconstructed blindly from adjacent timestamps.

The negative calculated values also indicate that timestamp ordering or trip sequencing contains additional anomalies that need to be accounted for before using timestamp-derived travel time for imputation.

In [21]:
same_row_relationship = df[
    df["travel_time_seconds"].notna()
    & df["move_timestamp"].notna()
    & df["stop_timestamp"].notna()
].copy()

same_row_relationship["calculated_travel_time"] = (
    same_row_relationship["stop_timestamp"]
    - same_row_relationship["move_timestamp"]
)

same_row_relationship[
    ["travel_time_seconds", "calculated_travel_time"]
].describe()

,travel_time_seconds,calculated_travel_time
count,43290.000000,43290.000000
mean,83.292261,83.292261
std,82.249213,82.249213
min,2.000000,2.000000
25%,42.000000,42.000000
50%,70.000000,70.000000
75%,102.000000,102.000000
max,3405.000000,3405.000000


### Finding: Same-Row Timestamps Exactly Reproduce Observed Travel Time

A second reconstruction test was performed using only rows where `travel_time_seconds`, `move_timestamp`, and `stop_timestamp` are all present.

For these **43,290 complete rows**, `stop_timestamp - move_timestamp` exactly matches `travel_time_seconds`:

- Mean: **83.29 s** for both measures
- Median: **70 s** for both measures
- 25th percentile: **42 s** for both
- 75th percentile: **102 s** for both
- Minimum: **2 s** for both
- Maximum: **3,405 s** for both
- Standard deviation: **82.25 s** for both

This confirms that, when all three fields are present, `travel_time_seconds` is directly consistent with the same-row timestamp difference.

Therefore, the earlier discrepancy found in Block 17 is specifically associated with using the **previous stop's timestamp** rather than the timestamps from the same row. This gives us a reliable basis for interpreting the five missing travel-time exceptions: the relevant question is whether their own `move_timestamp` and `stop_timestamp` form a valid pair—not whether travel time can be inferred from the previous stop.

In [22]:
timestamp_difference_check = df[
    df["move_timestamp"].notna()
    & df["stop_timestamp"].notna()
].copy()

timestamp_difference_check["timestamp_difference"] = (
    timestamp_difference_check["stop_timestamp"]
    - timestamp_difference_check["move_timestamp"]
)

timestamp_difference_check[
    timestamp_difference_check["timestamp_difference"] < 0
][
    [
        "move_timestamp",
        "stop_timestamp",
        "timestamp_difference",
        "travel_time_seconds",
        "route_id",
        "stop_id",
        "trip_id",
    ]
]

,move_timestamp,stop_timestamp,timestamp_difference,travel_time_seconds,route_id,stop_id,trip_id
5877,1.568808e+09,1.568808e+09,-6.0,NaN,Green-C,70237,39988633
20711,1.568829e+09,1.568829e+09,-45.0,NaN,Green-D,70161,39989564
23810,1.568834e+09,1.568834e+09,-76.0,NaN,Red,70093,ADDED-1568817507
34765,1.568849e+09,1.568849e+09,-33.0,NaN,Red,70093,ADDED-1568817722
36791,1.568852e+09,1.568852e+09,-431.0,NaN,Red,70097,41527514


### Finding: All Five Missing Travel-Time Rows Have Invalid Timestamp Differences

The five rows with missing `travel_time_seconds` were examined using the available `move_timestamp` and `stop_timestamp` values.

For all five rows, the calculated timestamp difference (`stop_timestamp - move_timestamp`) is **negative**:

- Green-C: **−6 seconds**
- Green-D: **−45 seconds**
- Red: **−76 seconds**
- Red: **−33 seconds**
- Red: **−431 seconds**

A valid travel time should represent elapsed time between movement and arrival at the stop, so these negative differences cannot be used as reliable replacements for the missing `travel_time_seconds`.

This provides an important finding: **although timestamps are present for all five missing travel-time rows, their timestamp ordering is invalid.** Therefore, timestamp-based reconstruction should not be used to impute these five missing travel-time values.

In [23]:
dwell_missing_profile = (
    df.groupby(
        [
            df["dwell_time_seconds"].isna().rename("dwell_time_missing"),
            df["move_timestamp"].isna().rename("move_timestamp_missing"),
            df["stop_timestamp"].isna().rename("stop_timestamp_missing"),
        ]
    )
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

dwell_missing_profile

,dwell_time_missing,move_timestamp_missing,stop_timestamp_missing,row_count
0,False,False,False,39776
3,True,False,False,3519
2,False,True,False,1345
5,True,True,False,204
4,True,False,True,95
1,False,False,True,31


### Dwell Time Missingness

The missingness pattern for `dwell_time_seconds` was compared with the availability of `move_timestamp` and `stop_timestamp`.

Of the 3,818 records where `dwell_time_seconds` is missing:

- 3,519 records have both `move_timestamp` and `stop_timestamp` present.
- 204 records have `move_timestamp` missing while `stop_timestamp` is present.
- 95 records have `stop_timestamp` missing while `move_timestamp` is present.

This shows that most missing `dwell_time_seconds` values occur **despite both operational timestamps being available**. Therefore, missing dwell time is not primarily explained by missing timestamp data.

A smaller number of records also show the opposite pattern: timestamps may be missing while `dwell_time_seconds` is present. This indicates that the relationship between dwell-time availability and timestamp completeness is not one-to-one.

The 3,519 records with both timestamps present warrant further investigation to determine whether dwell time can be independently derived from the timestamp relationship. No imputation or modification of `dwell_time_seconds` is performed at this stage.

In [24]:
dwell_missing_with_timestamps = df[
    df["dwell_time_seconds"].isna()
    & df["move_timestamp"].notna()
    & df["stop_timestamp"].notna()
].copy()

dwell_missing_with_timestamps[
    [
        "move_timestamp",
        "stop_timestamp",
        "dwell_time_seconds",
        "route_id",
        "stop_id",
        "trip_id",
    ]
].assign(
    timestamp_difference=lambda x:
        x["stop_timestamp"] - x["move_timestamp"]
)

,move_timestamp,stop_timestamp,dwell_time_seconds,route_id,stop_id,trip_id,timestamp_difference
6,1.568794e+09,1.568794e+09,NaN,Green-C,71199,ADDED-1568746064,90.0
7,1.568794e+09,1.568794e+09,NaN,Green-C,70196,ADDED-1568746065,49.0
20,1.568795e+09,1.568798e+09,NaN,Red,70094,ADDED-1568746069,3048.0
29,1.568796e+09,1.568796e+09,NaN,Green-C,70237,ADDED-1568746065,29.0
32,1.568796e+09,1.568799e+09,NaN,Red,70094,ADDED-1568746072,3182.0
...,...,...,...,...,...,...,...
44952,1.568871e+09,1.568871e+09,NaN,Mattapan,70261,ADDED-1568818152,384.0
44955,1.568871e+09,1.568871e+09,NaN,Red,70061,ADDED-1568818099,121.0
44961,1.568871e+09,1.568871e+09,NaN,Green-B,70107,ADDED-1568818116,144.0
44963,1.568871e+09,1.568871e+09,NaN,Green-D,70161,39988545,66.0


### Dwell Time with Complete Timestamps

Among the 3,818 records with missing `dwell_time_seconds`, 3,519 records have both `move_timestamp` and `stop_timestamp` present.

The timestamp difference was calculated for these records as:

`stop_timestamp - move_timestamp`

The resulting differences vary considerably, ranging from short intervals such as 29–90 seconds to much larger values exceeding 3,000 seconds.

Because the derived timestamp difference does not show a simple or obviously consistent pattern, it should not be assumed to represent `dwell_time_seconds` directly. The timestamps may capture different operational events or intervals than the stored dwell-time measure.

Therefore, the timestamp difference is treated as a diagnostic variable rather than an imputation rule. No missing dwell-time values are reconstructed at this stage.

In [25]:
dwell_route_profile = (
    df.groupby("route_id")["dwell_time_seconds"]
    .agg(
        total_rows="size",
        missing_rows=lambda s: s.isna().sum(),
        non_missing_rows=lambda s: s.notna().sum()
    )
)

dwell_route_profile["missing_percent"] = (
    dwell_route_profile["missing_rows"]
    / dwell_route_profile["total_rows"]
    * 100
).round(2)

dwell_route_profile.sort_values("missing_percent", ascending=False)

,total_rows,missing_rows,non_missing_rows,missing_percent
route_id,,,,
Mattapan,1605,245,1360,15.26
Green-D,5465,571,4894,10.45
Red,7600,784,6816,10.32
Blue,4327,437,3890,10.10
Green-C,6293,530,5763,8.42
Green-E,6050,427,5623,7.06
Orange,6019,366,5653,6.08
Green-B,7611,458,7153,6.02


### Dwell Time Missingness by Route

The distribution of missing `dwell_time_seconds` values was examined across routes to determine whether missingness is concentrated within particular services.

The percentage of missing dwell-time values varies noticeably by route:

- **Mattapan:** 15.26% (245 of 1,605 rows)
- **Green-D:** 10.45% (571 of 5,465 rows)
- **Red:** 10.32% (784 of 7,600 rows)
- **Blue:** 10.10% (437 of 4,327 rows)
- **Green-C:** 8.42% (530 of 6,293 rows)
- **Green-E:** 7.06% (427 of 6,050 rows)
- **Orange:** 6.08% (366 of 6,019 rows)
- **Green-B:** 6.02% (458 of 7,611 rows)

The highest missingness occurs on the **Mattapan route**, where approximately 15.26% of dwell-time observations are missing. The remaining routes range from approximately 6% to 10.5%.

This indicates that dwell-time missingness has a **route-dependent pattern**, rather than being uniformly distributed across the dataset. However, route-level differences alone do not establish the cause of the missingness.

No imputation or row removal is performed at this stage.

In [26]:
dwell_stop_profile = (
    df.groupby(["route_id", "stop_id"])["dwell_time_seconds"]
    .agg(
        total_rows="size",
        missing_rows=lambda s: s.isna().sum(),
        non_missing_rows=lambda s: s.notna().sum()
    )
)

dwell_stop_profile["missing_percent"] = (
    dwell_stop_profile["missing_rows"]
    / dwell_stop_profile["total_rows"]
    * 100
).round(2)

dwell_stop_profile = dwell_stop_profile[
    dwell_stop_profile["missing_rows"] > 0
].sort_values(
    "missing_percent",
    ascending=False
)

dwell_stop_profile

total_rows  missing_rows  non_missing_rows  missing_percent
route_id stop_id                                                             
Green-B  70107           175           175                 0           100.00
Blue     70060           162           162                 0           100.00
         70838           190           190                 0           100.00
Green-C  70209             6             6                 0           100.00
         70196             1             1                 0           100.00
...                      ...           ...               ...              ...
Red      70083           216             1               215             0.46
         70074           216             1               215             0.46
         70071           216             1               215             0.46
         70070           216             1               215             0.46
         70080           216             1               215             0.46

[208 rows x 4 columns]

### Dwell Time Missingness by Stop

Dwell-time missingness was examined at the route-and-stop level to determine whether the route-level differences observed above were concentrated at particular stops.

The results show substantial variation in missingness across individual stops. Several stops have **100% missing dwell-time observations**, including:

- Green-B stop `70107`: 175 of 175 rows missing.
- Blue stop `70060`: 162 of 162 rows missing.
- Blue stop `70838`: 190 of 190 rows missing.

Other stops have only isolated missing values. For example, several Red stops show 1 missing observation out of 216, corresponding to only 0.46% missingness.

This concentration of missingness at particular stops suggests that at least part of the missing `dwell_time_seconds` pattern may be **structural or stop-specific**, rather than randomly distributed across the dataset.

The 100% missingness rates should be interpreted together with the number of observations at each stop, since some stops have very small row counts.

No imputation or row removal is performed at this stage.

In [27]:
dwell_fully_missing_stops = dwell_stop_profile[
    dwell_stop_profile["missing_percent"] == 100
].sort_values(
    "missing_rows",
    ascending=False
)

dwell_fully_missing_stops

,,total_rows,missing_rows,non_missing_rows,missing_percent
route_id,stop_id,,,,
Red,70078,216,216,0,100.0
Blue,70838,190,190,0,100.0
Green-B,70107,175,175,0,100.0
Blue,70060,162,162,0,100.0
Green-E,70209,156,156,0,100.0
Green-C,70237,151,151,0,100.0
Green-D,70161,130,130,0,100.0
Red,70093,121,121,0,100.0
Mattapan,70275,110,110,0,100.0


### Fully Missing Dwell Time at Specific Stops

The route-and-stop analysis was further restricted to combinations where `dwell_time_seconds` is missing for **100% of observations**.

Several route/stop combinations show substantial complete missingness. The largest examples include:

- Red / stop `70078`: 216 of 216 observations missing.
- Blue / stop `70838`: 190 of 190 observations missing.
- Green-B / stop `70107`: 175 of 175 observations missing.
- Blue / stop `70060`: 162 of 162 observations missing.
- Green-E / stop `70209`: 156 of 156 observations missing.
- Green-C / stop `70237`: 151 of 151 observations missing.
- Green-D / stop `70161`: 130 of 130 observations missing.
- Red / stop `70093`: 121 of 121 observations missing.
- Mattapan / stop `70275`: 110 of 110 observations missing.

This provides stronger evidence that dwell-time missingness is **structurally concentrated in certain route/stop combinations** rather than being uniformly distributed.

Several additional combinations have 100% missingness but very small numbers of observations. These are less informative and should not be treated as equivalent to the larger groups.

The concentration of missingness at specific stops suggests that the availability of `dwell_time_seconds` may depend on the operational characteristics or recording behavior associated with particular stops. The cause is not established from this analysis alone and requires further investigation.

No imputation or row removal is performed at this stage.

In [28]:
dwell_structural_check = (
    df.groupby(["route_id", "stop_id"])
    .agg(
        total_rows=("dwell_time_seconds", "size"),
        dwell_missing=("dwell_time_seconds", lambda s: s.isna().sum()),
        move_timestamp_missing=("move_timestamp", lambda s: s.isna().sum()),
        stop_timestamp_missing=("stop_timestamp", lambda s: s.isna().sum()),
    )
)

dwell_structural_check["dwell_missing_percent"] = (
    dwell_structural_check["dwell_missing"]
    / dwell_structural_check["total_rows"]
    * 100
).round(2)

dwell_structural_check = dwell_structural_check[
    dwell_structural_check["dwell_missing_percent"] == 100
].sort_values(
    "total_rows",
    ascending=False
)

dwell_structural_check

,,total_rows,dwell_missing,move_timestamp_missing,stop_timestamp_missing,dwell_missing_percent
route_id,stop_id,,,,,
Red,70078,216,216,1,0,100.0
Blue,70838,190,190,0,0,100.0
Green-B,70107,175,175,6,0,100.0
Blue,70060,162,162,0,0,100.0
Green-E,70209,156,156,0,0,100.0
Green-C,70237,151,151,4,0,100.0
Green-D,70161,130,130,4,0,100.0
Red,70093,121,121,15,0,100.0
Mattapan,70275,110,110,3,0,100.0


### Relationship Between Dwell-Time Missingness and Timestamps

The fully missing route/stop combinations were examined to determine whether `dwell_time_seconds` is missing because the underlying operational timestamps are also unavailable.

For the largest fully missing groups, `stop_timestamp` is consistently present across all observations. For example:

- Red / stop `70078`: 216 of 216 dwell values missing, but only 1 `move_timestamp` is missing and **0 `stop_timestamp` values are missing**.
- Blue / stop `70838`: 190 of 190 dwell values missing, with **0 missing move or stop timestamps**.
- Green-B / stop `70107`: 175 of 175 dwell values missing, with 6 missing move timestamps but **0 missing stop timestamps**.
- Blue / stop `70060`: 162 of 162 dwell values missing, with **0 missing move or stop timestamps**.
- Green-E / stop `70209`: 156 of 156 dwell values missing, with **0 missing move or stop timestamps**.
- Green-C / stop `70237`: 151 of 151 dwell values missing, with 4 missing move timestamps but **0 missing stop timestamps**.
- Green-D / stop `70161`: 130 of 130 dwell values missing, with 4 missing move timestamps but **0 missing stop timestamps**.
- Red / stop `70093`: 121 of 121 dwell values missing, with 15 missing move timestamps but **0 missing stop timestamps**.

This shows that complete dwell-time missingness cannot be explained simply by missing timestamps. In several substantial route/stop groups, both timestamps are available while `dwell_time_seconds` remains entirely missing.

Therefore, `dwell_time_seconds` appears to have an additional **structural or field-specific missingness pattern** associated with particular route/stop combinations.

No imputation or row removal is performed at this stage.

In [29]:
dwell_relationship = df.sort_values(
    ["trip_id", "stop_sequence"]
).copy()

dwell_relationship["next_move_timestamp"] = (
    dwell_relationship.groupby("trip_id")["move_timestamp"].shift(-1)
)

dwell_relationship["calculated_dwell_time"] = (
    dwell_relationship["next_move_timestamp"]
    - dwell_relationship["stop_timestamp"]
)

dwell_relationship[
    [
        "dwell_time_seconds",
        "calculated_dwell_time"
    ]
].dropna().describe()

,dwell_time_seconds,calculated_dwell_time
count,41070.000000,41070.000000
mean,100.507548,62.744266
std,622.785243,75.538841
min,1.000000,-1586.000000
25%,40.000000,39.000000
50%,51.000000,50.000000
75%,67.000000,65.000000
max,53577.000000,4971.000000


### Dwell-Time Relationship to Adjacent Timestamps

To determine whether `dwell_time_seconds` can be reconstructed from the operational timestamps, the next stop's `move_timestamp` was compared with the current stop's `stop_timestamp` within each `trip_id`.

For the 41,070 records where both `dwell_time_seconds` and the calculated value are available, the distributions are similar around the median:

- `dwell_time_seconds`: median = 51 seconds, 25th percentile = 40 seconds, 75th percentile = 67 seconds.
- `calculated_dwell_time`: median = 50 seconds, 25th percentile = 39 seconds, 75th percentile = 65 seconds.

However, the overall distributions are not equivalent. The observed `dwell_time_seconds` contains extreme values, including a maximum of 53,577 seconds, while the calculated measure has a maximum of 4,971 seconds. The calculated measure also contains negative values, with a minimum of -1,586 seconds.

This indicates that the timestamp-derived measure broadly follows the typical scale of observed dwell times but also exposes timing anomalies and differences in how the original dwell-time field was recorded.

Therefore, the timestamp relationship is potentially useful for further investigation, but it is **not yet sufficient to justify reconstructing or imputing missing dwell times**.

No imputation or row removal is performed at this stage.

In [30]:
dwell_comparison = dwell_relationship[
    [
        "dwell_time_seconds",
        "calculated_dwell_time"
    ]
].dropna().copy()

dwell_comparison["dwell_difference"] = (
    dwell_comparison["dwell_time_seconds"]
    - dwell_comparison["calculated_dwell_time"]
)

dwell_comparison["dwell_difference"].describe()

count    41070.000000
mean        37.763282
std        615.768520
min       -357.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      53553.000000
Name: dwell_difference, dtype: float64

### Validation of Timestamp-Derived Dwell Time

The difference between the recorded `dwell_time_seconds` and the timestamp-derived `calculated_dwell_time` was examined for the 41,070 records where both measures are available.

The comparison shows strong agreement for the majority of observations:

- 25th percentile of the difference: 0 seconds
- Median difference: 0 seconds
- 75th percentile of the difference: 0 seconds

This means that at least 75% of the observed records have a difference of zero or less, with the central portion of the distribution concentrated exactly at zero.

However, the mean difference is 37.76 seconds and the range is extremely wide, from −357 seconds to 53,553 seconds. These extreme values indicate that a smaller subset of records has substantial disagreement between the recorded dwell time and the timestamp-derived measure.

Therefore, the timestamp relationship appears highly consistent for the majority of records but is affected by a small number of substantial discrepancies. The extreme differences should be investigated before using the timestamp-derived value to reconstruct missing `dwell_time_seconds`.

No imputation or row removal is performed at this stage.

In [31]:
dwell_comparison["exact_match"] = (
    dwell_comparison["dwell_difference"] == 0
)

dwell_comparison["exact_match"].value_counts()

exact_match
True     38628
False     2442
Name: count, dtype: int64

### Exact Agreement Between Recorded and Timestamp-Derived Dwell Time

The recorded `dwell_time_seconds` was compared with the dwell time calculated from the timestamp relationship.

Among the 41,070 records where both measures were available:

- 38,628 records (94.05%) were exact matches.
- 2,442 records (5.95%) did not match.

This indicates a strong relationship between the recorded dwell time and the operational timestamps. The timestamp-derived measure reproduces the recorded value exactly for approximately 94% of comparable observations.

The remaining 5.95% of records require additional investigation because the discrepancy may reflect unusual operational events, timestamp irregularities, or differences in how dwell time was originally calculated.

At this stage, the result supports investigating timestamp-based reconstruction of missing dwell times, but does not yet justify automatically imputing all missing values.

In [32]:
dwell_comparison.loc[
    ~dwell_comparison["exact_match"],
    "dwell_difference"
].abs().describe()

count     2442.000000
mean       638.791155
std       2448.508545
min          1.000000
25%         23.000000
50%        223.000000
75%        465.000000
max      53553.000000
Name: dwell_difference, dtype: float64

### Magnitude of Dwell-Time Discrepancies

The 2,442 records where the recorded `dwell_time_seconds` did not exactly match the timestamp-derived value were examined using the absolute difference between the two measures.

The discrepancies vary substantially:

- Minimum absolute difference: 1 second
- 25th percentile: 23 seconds
- Median: 223 seconds
- 75th percentile: 465 seconds
- Maximum: 53,553 seconds
- Mean: 638.79 seconds

The median discrepancy of 223 seconds shows that the non-matching records generally differ by more than a small rounding or measurement adjustment. The very large maximum also indicates the presence of substantial outliers.

Combined with the 94.05% exact-match rate established previously, this suggests that timestamp-derived dwell time is highly reliable for most records but does not reproduce the recorded field consistently for a smaller subset.

The non-matching records therefore require further investigation before any timestamp-based reconstruction is applied to missing `dwell_time_seconds`.

No imputation or row removal is performed at this stage.

In [33]:
dwell_nonmatch_route = (
    dwell_relationship[
        dwell_relationship["dwell_time_seconds"].notna()
        & dwell_relationship["calculated_dwell_time"].notna()
        & (
            dwell_relationship["dwell_time_seconds"]
            != dwell_relationship["calculated_dwell_time"]
        )
    ]
    .groupby("route_id")
    .size()
    .sort_values(ascending=False)
)

dwell_nonmatch_route

route_id
Red         379
Green-B     366
Blue        345
Green-D     302
Green-C     297
Orange      280
Green-E     271
Mattapan    202
dtype: int64

### Distribution of Dwell-Time Discrepancies by Route

The 2,442 non-matching records were grouped by `route_id` to determine whether the discrepancies were concentrated on a particular route.

The number of non-matching records by route is:

- Red: 379
- Green-B: 366
- Blue: 345
- Green-D: 302
- Green-C: 297
- Orange: 280
- Green-E: 271
- Mattapan: 202

All routes contain non-matching records. The highest count occurs on the Red Line (379 records), while Mattapan has the lowest count (202 records).

These are raw discrepancy counts and should not be interpreted as evidence that a particular route has a higher discrepancy rate, because the routes contain different numbers of observations. A route-level rate is therefore needed before drawing conclusions about whether discrepancies are disproportionately concentrated on any route.

No imputation or row removal is performed at this stage.

In [34]:
dwell_route_comparison = (
    dwell_relationship[
        dwell_relationship["dwell_time_seconds"].notna()
        & dwell_relationship["calculated_dwell_time"].notna()
    ]
    .groupby("route_id")
    .agg(
        comparable_rows=("dwell_time_seconds", "size"),
        non_matching_rows=(
            "dwell_time_seconds",
            lambda s: (
                s
                != dwell_relationship.loc[s.index, "calculated_dwell_time"]
            ).sum()
        )
    )
)

dwell_route_comparison["non_matching_percent"] = (
    dwell_route_comparison["non_matching_rows"]
    / dwell_route_comparison["comparable_rows"]
    * 100
).round(2)

dwell_route_comparison.sort_values(
    "non_matching_percent",
    ascending=False
)

,comparable_rows,non_matching_rows,non_matching_percent
route_id,,,
Mattapan,1349,202,14.97
Blue,3870,345,8.91
Green-D,4873,302,6.20
Red,6814,379,5.56
Green-C,5757,297,5.16
Green-B,7143,366,5.12
Orange,5650,280,4.96
Green-E,5614,271,4.83


### Rate of Dwell-Time Discrepancies by Route

Because the routes contain different numbers of comparable observations, the non-matching records were converted into route-level percentages.

The discrepancy rate varies substantially by route:

- Mattapan: 14.97%
- Blue: 8.91%
- Green-D: 6.20%
- Red: 5.56%
- Green-C: 5.16%
- Green-B: 5.12%
- Orange: 4.96%
- Green-E: 4.83%

Mattapan has the highest non-matching rate at 14.97%, noticeably above the other routes. Blue has the second-highest rate at 8.91%, while the remaining routes range from 4.83% to 6.20%.

This indicates that the disagreement between recorded and timestamp-derived dwell time is not uniformly distributed across routes. In particular, Mattapan warrants additional investigation before timestamp-based reconstruction is considered.

The route-level discrepancy rate does not by itself establish why the differences occur. No imputation or row removal is performed at this stage.

In [35]:
dwell_mismatch_magnitude_by_route = (
    dwell_relationship[
        dwell_relationship["dwell_time_seconds"].notna()
        & dwell_relationship["calculated_dwell_time"].notna()
        & (
            dwell_relationship["dwell_time_seconds"]
            != dwell_relationship["calculated_dwell_time"]
        )
    ]
    .assign(
        absolute_dwell_difference=lambda x: (
            x["dwell_time_seconds"]
            - x["calculated_dwell_time"]
        ).abs()
    )
    .groupby("route_id")["absolute_dwell_difference"]
    .agg(
        mismatch_count="size",
        median_difference="median",
        mean_difference="mean",
        max_difference="max"
    )
    .sort_values("median_difference", ascending=False)
    .round(2)
)

dwell_mismatch_magnitude_by_route

,mismatch_count,median_difference,mean_difference,max_difference
route_id,,,,
Green-C,297,449.0,968.12,53553.0
Green-B,366,335.0,1079.29,39070.0
Blue,345,256.0,306.49,1626.0
Green-E,271,239.0,528.07,16062.0
Green-D,302,180.0,1054.42,28682.0
Mattapan,202,176.5,496.51,20718.0
Red,379,16.0,372.52,19910.0
Orange,280,10.0,245.03,15673.0


### Magnitude of Dwell-Time Discrepancies by Route

The magnitude of the non-matching dwell-time differences was examined by route to determine whether routes with higher mismatch rates also had larger discrepancies.

The results show that discrepancy magnitude varies considerably across routes:

- Green-C has the highest median absolute difference at 449 seconds.
- Green-B follows with a median difference of 335 seconds.
- Blue, Green-E, and Green-D have median differences of 256, 239, and 180 seconds respectively.
- Mattapan has a median difference of 176.5 seconds despite having the highest mismatch rate identified previously.
- Red and Orange have substantially smaller median differences of 16 and 10 seconds respectively.

Several routes also contain very large maximum discrepancies. Green-C has a maximum difference of 53,553 seconds, Green-B 39,070 seconds, and Green-D 28,682 seconds.

These results show that mismatch frequency and mismatch magnitude are different aspects of the data-quality issue. Mattapan has the highest proportion of non-matching records, but other routes exhibit substantially larger discrepancies when mismatches occur.

Therefore, the discrepancies cannot be treated as a single uniform route-level issue. Further investigation should focus on whether the timestamp-derived calculation can reliably reconstruct the **missing** dwell-time records, rather than assuming that all missing values should be imputed.

No imputation or row removal is performed at this stage.

In [36]:
dwell_missing_with_timestamps = dwell_relationship[
    dwell_relationship["dwell_time_seconds"].isna()
    & dwell_relationship["stop_timestamp"].notna()
    & dwell_relationship["next_move_timestamp"].notna()
].copy()

dwell_missing_with_timestamps["calculated_dwell_time"] = (
    dwell_missing_with_timestamps["next_move_timestamp"]
    - dwell_missing_with_timestamps["stop_timestamp"]
)

dwell_missing_with_timestamps[
    [
        "dwell_time_seconds",
        "stop_timestamp",
        "next_move_timestamp",
        "calculated_dwell_time",
        "route_id",
        "stop_id",
        "trip_id"
    ]
]

,dwell_time_seconds,stop_timestamp,next_move_timestamp,calculated_dwell_time,route_id,stop_id,trip_id
68,NaN,1.568798e+09,1.568798e+09,99.0,Green-D,70162,39988441
295,NaN,1.568799e+09,1.568799e+09,45.0,Green-D,70162,39988444
721,NaN,1.568800e+09,1.568800e+09,24.0,Green-D,70176,39988446
1009,NaN,1.568801e+09,1.568801e+09,36.0,Green-D,70176,39988447
1048,NaN,1.568801e+09,1.568801e+09,12.0,Green-C,70234,39988450
...,...,...,...,...,...,...,...
44045,NaN,1.568867e+09,1.568867e+09,45.0,Red,70092,ADDED-1568818123
44845,NaN,1.568870e+09,1.568870e+09,-19.0,Green-E,70242,ADDED-1568818133
2487,NaN,1.568803e+09,1.568803e+09,56.0,Blue,70059,NONREV-1568745449
24448,NaN,1.568835e+09,1.568835e+09,166.0,Red,70094,NONREV-1568817598


### Missing Dwell Time with Recoverable Timestamp Information

The records where `dwell_time_seconds` is missing were examined to determine whether the value can be calculated from adjacent operational timestamps.

A total of **512 missing dwell-time records** have both:

- a non-missing `stop_timestamp` for the current stop, and
- a non-missing `next_move_timestamp` from the following stop within the same trip.

For these records, a timestamp-derived dwell time can be calculated as:

`next_move_timestamp - stop_timestamp`

These 512 records represent approximately 14.55% of the 3,519 records where `dwell_time_seconds` is missing.

The calculated values are generally positive and appear within plausible dwell-time ranges in the displayed observations. However, at least one record produces a negative calculated dwell time (−19 seconds), indicating that timestamp inconsistencies are also present among the potentially recoverable records.

Therefore, the 512 records should not yet be automatically imputed. The calculated dwell times must first be validated for negative values and other implausible durations.

In [37]:
dwell_missing_with_timestamps["dwell_category"] = (
    dwell_missing_with_timestamps["calculated_dwell_time"]
    .apply(
        lambda x: (
            "negative"
            if x < 0
            else "zero"
            if x == 0
            else "positive"
        )
    )
)

dwell_missing_with_timestamps["dwell_category"].value_counts()

dwell_category
positive    347
negative    159
zero          6
Name: count, dtype: int64

### Validation of Timestamp-Derived Values for Missing Dwell Time

The 512 records with missing `dwell_time_seconds` and sufficient timestamp information were classified according to their timestamp-derived dwell duration.

The results are:

- 347 records (67.77%) have positive calculated dwell times.
- 159 records (31.05%) have negative calculated dwell times.
- 6 records (1.17%) have a calculated dwell time of exactly zero.

The large number of negative values is a significant data-quality issue. A negative dwell duration is not a plausible operational dwell time and indicates that the timestamp sequence is inconsistent for a substantial portion of the potentially recoverable records.

Although 347 records produce positive calculated values, the presence of negative and zero values means that timestamp availability alone is not sufficient to determine whether a missing dwell value can be safely reconstructed.

The positive calculated values therefore require further validation before any imputation decision is made. No values are imputed or removed at this stage.

In [38]:
dwell_negative_by_route = (
    dwell_missing_with_timestamps[
        dwell_missing_with_timestamps["calculated_dwell_time"] < 0
    ]
    .groupby("route_id")
    .size()
    .sort_values(ascending=False)
)

dwell_negative_by_route

route_id
Green-D     45
Green-C     31
Green-E     25
Red         18
Green-B     13
Mattapan    12
Orange       8
Blue         7
dtype: int64

### Distribution of Negative Calculated Dwell Times by Route

The 159 records with negative timestamp-derived dwell times were grouped by `route_id` to determine whether the timestamp inconsistency was concentrated on a particular route.

The negative cases are distributed across all routes:

- Green-D: 45
- Green-C: 31
- Green-E: 25
- Red: 18
- Green-B: 13
- Mattapan: 12
- Orange: 8
- Blue: 7

Green-D contains the largest number of negative calculated dwell times (45 records), followed by Green-C (31) and Green-E (25). Blue has the fewest negative cases (7).

Because these are raw counts, they should not be interpreted as route-level rates without accounting for the number of missing dwell-time records on each route.

The presence of negative calculated dwell times across every route indicates that the timestamp inconsistency is not isolated to a single route. This further supports treating timestamp-derived dwell values as candidate reconstructions requiring validation rather than automatically imputing all missing values.

No imputation or row removal is performed at this stage.

In [39]:
dwell_missing_route_quality = (
    dwell_missing_with_timestamps
    .groupby("route_id")
    .agg(
        timestamp_available_rows=("calculated_dwell_time", "size"),
        positive_rows=(
            "calculated_dwell_time",
            lambda s: (s > 0).sum()
        ),
        zero_rows=(
            "calculated_dwell_time",
            lambda s: (s == 0).sum()
        ),
        negative_rows=(
            "calculated_dwell_time",
            lambda s: (s < 0).sum()
        )
    )
)

dwell_missing_route_quality["negative_percent"] = (
    dwell_missing_route_quality["negative_rows"]
    / dwell_missing_route_quality["timestamp_available_rows"]
    * 100
).round(2)

dwell_missing_route_quality.sort_values(
    "negative_percent",
    ascending=False
)

,timestamp_available_rows,positive_rows,zero_rows,negative_rows,negative_percent
route_id,,,,,
Mattapan,21,5,4,12,57.14
Green-E,70,43,2,25,35.71
Green-D,137,92,0,45,32.85
Red,57,39,0,18,31.58
Green-C,107,76,0,31,28.97
Blue,27,20,0,7,25.93
Orange,31,23,0,8,25.81
Green-B,62,49,0,13,20.97


### Route-Level Quality of Timestamp-Derived Dwell Times

The timestamp-derived dwell times were evaluated by route to determine the proportion of candidate values that produce negative elapsed times.

The negative rate varies substantially across routes:

- Mattapan has the highest negative rate at **57.14%** (12 of 21 timestamp-available records).
- Green-E has a negative rate of **35.71%** (25 of 70).
- Green-D has a negative rate of **32.85%** (45 of 137).
- Red has a negative rate of **31.58%** (18 of 57).
- Green-C has a negative rate of **28.97%** (31 of 107).
- Blue and Orange are both around **26%**.
- Green-B has the lowest negative rate at **20.97%** (13 of 62).

These percentages should be interpreted cautiously because the number of timestamp-available records varies considerably by route. In particular, Mattapan has only 21 available records, so its 57.14% rate is based on a relatively small sample.

Overall, the negative timestamp-derived dwell values are not confined to one route and occur at meaningful rates across all routes. This indicates that reconstructing missing dwell times solely from the timestamp difference would introduce a substantial number of implausible values.

Therefore, timestamp-derived dwell time should be treated as a validation signal rather than an automatic replacement for missing `dwell_time_seconds`.

In [40]:
dwell_negative = dwell_missing_with_timestamps[
    dwell_missing_with_timestamps["calculated_dwell_time"] < 0
].copy()

dwell_negative[
    [
        "trip_id",
        "route_id",
        "stop_id",
        "stop_sequence",
        "stop_timestamp",
        "next_move_timestamp",
        "calculated_dwell_time",
    ]
].sort_values("calculated_dwell_time")

,trip_id,route_id,stop_id,stop_sequence,stop_timestamp,next_move_timestamp,calculated_dwell_time
39053,ADDED-1568817862,Green-D,70174,380,1.568856e+09,1.568855e+09,-1119.0
6464,39988586,Green-C,70236,190,1.568809e+09,1.568808e+09,-668.0
22051,ADDED-1568817462,Green-D,70178,400,1.568832e+09,1.568831e+09,-231.0
13244,ADDED-1568814061,Green-D,70157,100,1.568817e+09,1.568817e+09,-134.0
34192,39988515,Green-C,70236,190,1.568848e+09,1.568848e+09,-115.0
...,...,...,...,...,...,...,...
4074,ADDED-1568746270,Green-C,70150,550,1.568806e+09,1.568806e+09,-1.0
42131,39990267,Green-E,70207,640,1.568863e+09,1.568863e+09,-1.0
21550,39990189,Green-E,70252,480,1.568831e+09,1.568831e+09,-1.0
22822,41845497,Mattapan,70273,7,1.568833e+09,1.568833e+09,-1.0


### Negative Calculated Dwell Times

The 159 records with negative `calculated_dwell_time` were inspected to examine the underlying timestamp relationship and operational context.

The most negative values are substantially below zero, including a minimum of -1,119 seconds. The negative cases occur across multiple routes, including Green-C, Green-D, Green-E, and Mattapan, and include both regular trip IDs and `ADDED-` trips.

These records represent cases where `next_move_timestamp` occurs before `stop_timestamp`, producing a negative calculated dwell duration. This indicates a timestamp-ordering inconsistency rather than a valid negative dwell time.

The records are retained for further investigation rather than being corrected or removed at this stage.

In [41]:
dwell_negative["trip_type"] = (
    dwell_negative["trip_id"]
    .astype(str)
    .str.extract(r"^(ADDED|NONREV)", expand=False)
    .fillna("REGULAR")
)

dwell_negative["trip_type"].value_counts()

trip_type
REGULAR    115
ADDED       44
Name: count, dtype: int64

### Negative Dwell Times by Trip Type

The 159 negative `calculated_dwell_time` records were classified by trip type.

- 115 records are associated with regular trips.
- 44 records are associated with `ADDED` trips.
- No negative dwell-time records are associated with `NONREV` trips.

Therefore, the negative dwell-time anomalies occur primarily in regular trips, with a smaller subset occurring in added-service trips. The issue is not exclusive to a particular operational trip category.

In [42]:
dwell_negative["route_id"].value_counts()

route_id
Green-D     45
Green-C     31
Green-E     25
Red         18
Green-B     13
Mattapan    12
Orange       8
Blue         7
Name: count, dtype: int64

### Negative Dwell Times by Route

The 159 negative `calculated_dwell_time` records were examined by route.

The anomalies are concentrated most heavily on the Green Line branches:

- Green-D: 45 records
- Green-C: 31 records
- Green-E: 25 records
- Red: 18 records
- Green-B: 13 records
- Mattapan: 12 records
- Orange: 8 records
- Blue: 7 records

The three Green Line branches together account for 101 of the 159 negative dwell-time records, or approximately 63.5% of the total.

This indicates that the timestamp-ordering anomaly is disproportionately concentrated on the Green Line branches, although it is present across all routes in the dataset.

In [43]:
negative_dwell_route_profile = (
    dwell_negative.groupby("route_id")
    .size()
    .rename("negative_rows")
    .to_frame()
    .join(
        dwell_route_profile["total_rows"]
    )
)

negative_dwell_route_profile["negative_percent"] = (
    negative_dwell_route_profile["negative_rows"]
    / negative_dwell_route_profile["total_rows"]
    * 100
).round(2)

negative_dwell_route_profile.sort_values(
    "negative_percent",
    ascending=False
)

,negative_rows,total_rows,negative_percent
route_id,,,
Green-D,45,5465,0.82
Mattapan,12,1605,0.75
Green-C,31,6293,0.49
Green-E,25,6050,0.41
Red,18,7600,0.24
Green-B,13,7611,0.17
Blue,7,4327,0.16
Orange,8,6019,0.13


### Negative Dwell Times Relative to Route Volume

To account for differences in route size, the negative dwell-time counts were compared with total rows for each route.

Green-D has the highest proportion of negative `calculated_dwell_time` records at 0.82%, followed by Mattapan at 0.75% and Green-C at 0.49%. The remaining routes range from 0.13% to 0.41%.

Although the negative dwell-time anomalies are concentrated more heavily on some routes than others, they represent less than 1% of total rows on every route.

Therefore, the negative dwell-time cases appear to be a relatively small data-quality issue rather than a widespread problem across the dataset.

In [44]:
dwell_negative["calculated_dwell_time"].describe()

count     159.000000
mean      -47.396226
std       104.695340
min     -1119.000000
25%       -56.500000
50%       -28.000000
75%       -10.000000
max        -1.000000
Name: calculated_dwell_time, dtype: float64

### Magnitude of Negative Dwell-Time Anomalies

The magnitude of the 159 negative `calculated_dwell_time` values was examined.

The median negative dwell time is -28 seconds, with the middle 50% of values ranging from -56.5 to -10 seconds. The mean is -47.4 seconds, while the minimum reaches -1,119 seconds.

This shows that most negative dwell-time anomalies involve relatively small timestamp reversals, but a small number of records contain substantially larger discrepancies.

These values are treated as timestamp-ordering anomalies rather than valid negative dwell durations. No correction or removal is performed at this stage.

In [45]:
scheduled_travel_missing_profile = (
    df.assign(
        scheduled_travel_missing=df["scheduled_travel_time"].isna(),
        scheduled_arrival_missing=df["scheduled_arrival_time"].isna(),
        scheduled_departure_missing=df["scheduled_departure_time"].isna(),
    )
    .groupby(
        [
            "scheduled_travel_missing",
            "scheduled_arrival_missing",
            "scheduled_departure_missing",
        ]
    )
    .size()
    .rename("row_count")
    .reset_index()
)

scheduled_travel_missing_profile

,scheduled_travel_missing,scheduled_arrival_missing,scheduled_departure_missing,row_count
0,False,False,False,42712
1,True,False,False,2041
2,True,True,True,217


### Scheduled Travel Time Missingness and Scheduled Timestamp Availability

Missing `scheduled_travel_time` values were compared with the availability of scheduled arrival and departure timestamps.

There are 42,712 rows where all three scheduled timing fields are present, 2,041 rows where `scheduled_travel_time` is missing despite both scheduled timestamps being available, and 217 rows where all three fields are missing.

The 2,041 rows with available scheduled timestamps account for 90.39% of the 2,258 missing `scheduled_travel_time` values, while the remaining 217 rows (9.61%) also have both scheduled timestamps missing.

This indicates that most missing `scheduled_travel_time` values are not explained by missing scheduled timestamps. The available timestamps can therefore be used to investigate whether the recorded scheduled travel time follows a consistent relationship with them.

In [46]:
scheduled_check = df.sort_values(
    ["trip_id", "stop_sequence"]
).copy()

scheduled_check["previous_scheduled_departure"] = (
    scheduled_check.groupby("trip_id")["scheduled_departure_time"].shift(1)
)

scheduled_check["calculated_scheduled_travel_time"] = (
    scheduled_check["scheduled_arrival_time"]
    - scheduled_check["previous_scheduled_departure"]
)

scheduled_comparable = scheduled_check[
    scheduled_check["scheduled_travel_time"].notna()
    & scheduled_check["calculated_scheduled_travel_time"].notna()
].copy()

scheduled_comparable[
    [
        "scheduled_travel_time",
        "calculated_scheduled_travel_time",
    ]
].describe()

,scheduled_travel_time,calculated_scheduled_travel_time
count,41718.000000,41718.000000
mean,132.163095,132.729757
std,55.430378,65.825595
min,0.000000,-1440.000000
25%,120.000000,120.000000
50%,120.000000,120.000000
75%,180.000000,180.000000
max,480.000000,3240.000000


### Scheduled Travel Time and Timestamp-Derived Values

The observed `scheduled_travel_time` was compared with a value calculated from the previous stop's scheduled departure and the current stop's scheduled arrival for 41,718 comparable rows.

The observed and calculated values have very similar central distributions. The mean is 132.16 seconds for `scheduled_travel_time` and 132.73 seconds for the calculated value, while both have a median of 120 seconds and a 75th percentile of 180 seconds.

The calculated values also contain a wider range, from -1,440 to 3,240 seconds, compared with 0 to 480 seconds for the observed `scheduled_travel_time`. This indicates that the timestamp relationship broadly reflects the recorded scheduled travel time but also produces anomalous values.

The relationship therefore requires further comparison at the individual-record level before it can be considered suitable for reconstructing missing `scheduled_travel_time` values.

In [47]:
scheduled_comparable["scheduled_travel_difference"] = (
    scheduled_comparable["scheduled_travel_time"]
    - scheduled_comparable["calculated_scheduled_travel_time"]
)

scheduled_comparable["scheduled_travel_difference"].value_counts().head(10)

scheduled_travel_difference
 0.0       41595
-120.0        26
-180.0        21
-240.0         8
-300.0         7
-420.0         5
-1080.0        5
-360.0         4
-60.0          4
 960.0         3
Name: count, dtype: int64

### Exact Match Between Scheduled Travel Time and Timestamp-Derived Values

The difference between the observed `scheduled_travel_time` and `calculated_scheduled_travel_time` was calculated for the 41,718 comparable rows.

There are 41,595 rows where the difference is exactly 0 seconds, leaving 123 rows with non-zero differences. The most frequent discrepancies are -120 seconds in 26 rows, -180 seconds in 21 rows, and -240 seconds in 8 rows.

This means the timestamp-derived value matches the recorded `scheduled_travel_time` for approximately 99.71% of comparable records. The remaining mismatches are relatively few but include differences large enough to require further investigation.

The next step is to examine the magnitude of these non-zero differences.

In [48]:
scheduled_mismatches = scheduled_comparable[
    scheduled_comparable["scheduled_travel_difference"] != 0
].copy()

scheduled_mismatches["absolute_difference"] = (
    scheduled_mismatches["scheduled_travel_difference"].abs()
)

scheduled_mismatches["absolute_difference"].describe()

count     123.000000
mean      465.365854
std       450.333743
min        60.000000
25%       180.000000
50%       300.000000
75%       660.000000
max      3120.000000
Name: absolute_difference, dtype: float64

### Magnitude of Scheduled Travel Time Mismatches

The magnitude of the 123 non-matching `scheduled_travel_difference` values was examined using their absolute differences.

The median absolute difference is 300 seconds, with the middle 50% of values ranging from 180 to 660 seconds. The mean is 465.36 seconds, while the minimum is 60 seconds and the maximum reaches 3,120 seconds.

This shows that the mismatches are not limited to small differences and can include substantially larger discrepancies, despite representing only a small portion of the comparable records.

These mismatches require further investigation before the timestamp relationship is used to reconstruct missing `scheduled_travel_time` values.

In [49]:
scheduled_mismatch_by_route = (
    scheduled_comparable.assign(
        scheduled_travel_mismatch=(
            scheduled_comparable["scheduled_travel_difference"] != 0
        )
    )
    .groupby("route_id")["scheduled_travel_mismatch"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "mismatch_count", "count": "comparable_count"})
)

scheduled_mismatch_by_route["mismatch_pct"] = (
    scheduled_mismatch_by_route["mismatch_count"]
    / scheduled_mismatch_by_route["comparable_count"]
    * 100
)

scheduled_mismatch_by_route.sort_values(
    "mismatch_pct", ascending=False
)

,mismatch_count,comparable_count,mismatch_pct
route_id,,,
Green-D,62,5001,1.239752
Green-C,25,5850,0.427350
Green-E,15,5696,0.263343
Green-B,12,7107,0.168848
Red,7,7111,0.098439
Blue,1,3901,0.025634
Orange,1,5676,0.017618
Mattapan,0,1376,0.000000


### Scheduled Travel Time Mismatch Rate by Route

The 123 non-matching `scheduled_travel_time` values were compared across routes to determine whether the mismatches are concentrated in particular route groups.

Green-D has the highest mismatch rate, with 62 mismatches among 5,001 comparable rows (1.24%). Green-C has 25 mismatches (0.43%), followed by Green-E with 15 (0.26%) and Green-B with 12 (0.17%). Red has 7 mismatches (0.10%), while Blue and Orange each have 1 mismatch. Mattapan has no mismatches among its 1,376 comparable rows.

The mismatches are therefore concentrated on the Green routes, particularly Green-D, rather than being evenly distributed across the dataset.

The next step is to examine whether the magnitude of the mismatches also varies by route.

In [50]:
scheduled_mismatch_by_route_magnitude = (
    scheduled_mismatches
    .groupby("route_id")["absolute_difference"]
    .agg(["count", "median", "mean", "max"])
    .rename(
        columns={
            "count": "mismatch_count",
            "median": "median_absolute_difference",
            "mean": "mean_absolute_difference",
            "max": "max_absolute_difference",
        }
    )
    .sort_values("median_absolute_difference", ascending=False)
)

scheduled_mismatch_by_route_magnitude

,mismatch_count,median_absolute_difference,mean_absolute_difference,max_absolute_difference
route_id,,,,
Green-B,12,720.0,820.000000,3120.0
Green-C,25,420.0,532.800000,1440.0
Green-E,15,420.0,580.000000,1200.0
Green-D,62,240.0,380.322581,1800.0
Red,7,180.0,240.000000,480.0
Blue,1,60.0,60.000000,60.0
Orange,1,60.0,60.000000,60.0


### Scheduled Travel Time Mismatch Magnitude by Route

The magnitude of the non-matching `scheduled_travel_difference` values was compared across routes.

Green-B has the largest median absolute difference at 720 seconds, followed by Green-C and Green-E at 420 seconds. Green-D has the highest mismatch count at 62 but a lower median absolute difference of 240 seconds. The largest discrepancy occurs on Green-B at 3,120 seconds.

This shows that mismatch frequency and mismatch magnitude vary by route. Green-D contributes the most mismatches, while Green-B has the largest typical and maximum discrepancies.

The next step is to examine the direction of these mismatches by route to determine whether the calculated values are generally higher or lower than the recorded `scheduled_travel_time`.

In [51]:
scheduled_mismatch_direction_by_route = (
    scheduled_mismatches
    .groupby("route_id")["scheduled_travel_difference"]
    .agg(["count", "median", "mean", "min", "max"])
    .rename(
        columns={
            "count": "mismatch_count",
            "median": "median_difference",
            "mean": "mean_difference",
            "min": "minimum_difference",
            "max": "maximum_difference",
        }
    )
    .sort_values("median_difference")
)

scheduled_mismatch_direction_by_route

,mismatch_count,median_difference,mean_difference,minimum_difference,maximum_difference
route_id,,,,,
Green-E,15,-420.0,-372.000000,-1200.0,1200.0
Green-B,12,-180.0,-150.000000,-3120.0,1620.0
Red,7,-180.0,-240.000000,-480.0,-120.0
Green-D,62,-180.0,-225.483871,-1800.0,780.0
Green-C,25,-120.0,-19.200000,-1440.0,1200.0
Blue,1,-60.0,-60.000000,-60.0,-60.0
Orange,1,-60.0,-60.000000,-60.0,-60.0


### Direction of Scheduled Travel Time Mismatches

The direction of the `scheduled_travel_difference` values was compared across routes to determine whether the calculated values were consistently higher or lower than the recorded `scheduled_travel_time`.

Green-E has the most negative median difference at -420 seconds, followed by Green-B, Red, and Green-D at -180 seconds. Green-C has a median difference of -120 seconds, while Blue and Orange each have a single mismatch of -60 seconds.

The direction is not consistent across all routes. Green-B ranges from -3,120 to 1,620 seconds and Green-E ranges from -1,200 to 1,200 seconds, showing that some routes contain discrepancies in both directions. Red is the only route where all observed mismatches are negative.

The mismatch pattern therefore does not indicate a single systematic offset. The next step is to apply the validated timestamp relationship to the rows with missing `scheduled_travel_time` and determine how many missing values can be calculated.

In [52]:
scheduled_missing_candidates = scheduled_check[
    scheduled_check["scheduled_travel_time"].isna()
    & scheduled_check["calculated_scheduled_travel_time"].notna()
].copy()

scheduled_missing_candidates["calculated_scheduled_travel_sign"] = (
    scheduled_missing_candidates["calculated_scheduled_travel_time"]
    .apply(
        lambda x: "Positive" if x > 0
        else "Zero" if x == 0
        else "Negative"
    )
)

scheduled_missing_candidates["calculated_scheduled_travel_sign"].value_counts()

calculated_scheduled_travel_sign
Negative    4
Positive    1
Name: count, dtype: int64

### Missing Scheduled Travel Time with Calculable Values

Only 5 rows with missing `scheduled_travel_time` have a calculable timestamp-derived value. Of these, 4 produce negative calculated travel times and 1 produces a positive value.

The predominance of negative values is inconsistent with a valid scheduled travel duration and suggests that the underlying scheduled timestamps may contain ordering anomalies in these records.

These 5 records should therefore be inspected individually before considering any reconstruction of the missing `scheduled_travel_time` values.

In [53]:
scheduled_missing_candidates[
    [
        "route_id",
        "trip_id",
        "stop_id",
        "stop_sequence",
        "scheduled_arrival_time",
        "previous_scheduled_departure",
        "calculated_scheduled_travel_time",
    ]
].sort_values(
    ["route_id", "trip_id", "stop_sequence"]
)

,route_id,trip_id,stop_id,stop_sequence,scheduled_arrival_time,previous_scheduled_departure,calculated_scheduled_travel_time
32682,Green-B,ADDED-1568817669,70196,50,66900.0,66360.0,540.0
38598,Green-B,ADDED-1568817804,70197,60,76020.0,76740.0,-720.0
37881,Green-B,ADDED-1568817806,70197,60,74760.0,75180.0,-420.0
38379,Green-B,ADDED-1568817831,70197,60,75600.0,76200.0,-600.0
18095,Green-D,ADDED-1568817376,70202,40,45660.0,45780.0,-120.0


### Missing Scheduled Travel Time Candidate Records

The 5 rows with missing `scheduled_travel_time` and calculable timestamp-derived values were inspected individually.

All 5 records belong to `ADDED-` trips. Four produce negative calculated travel times of -720, -420, -600, and -120 seconds because the current `scheduled_arrival_time` occurs before the previous scheduled departure.

The remaining record produces a positive calculated value of 540 seconds. This is substantially larger than the typical scheduled travel times observed earlier, although it does not by itself establish that the value is invalid.

These records therefore require additional context before any of the 5 missing `scheduled_travel_time` values can be considered suitable for reconstruction.

In [55]:
scheduled_candidate_indices = scheduled_missing_candidates.index

scheduled_context = scheduled_check[
    scheduled_check["trip_id"].isin(
        scheduled_missing_candidates["trip_id"]
    )
].copy()

scheduled_context[
    [
        "route_id",
        "trip_id",
        "stop_id",
        "stop_sequence",
        "scheduled_arrival_time",
        "scheduled_departure_time",
        "scheduled_travel_time",
        "calculated_scheduled_travel_time",
    ]
].sort_values(
    ["trip_id", "stop_sequence"]
)

,route_id,trip_id,stop_id,stop_sequence,scheduled_arrival_time,scheduled_departure_time,scheduled_travel_time,calculated_scheduled_travel_time
18053,Green-C,ADDED-1568817376,70202,40,45780.0,45780.0,120.0,NaN
18095,Green-D,ADDED-1568817376,70202,40,45660.0,45660.0,NaN,-120.0
18130,Green-D,ADDED-1568817376,70198,70,45780.0,45780.0,120.0,120.0
18235,Green-D,ADDED-1568817376,70159,90,45960.0,45960.0,180.0,180.0
18322,Green-D,ADDED-1568817376,70157,100,46080.0,46080.0,120.0,120.0
...,...,...,...,...,...,...,...,...
39518,Green-B,ADDED-1568817831,70117,270,77820.0,77820.0,60.0,60.0
39570,Green-B,ADDED-1568817831,70115,280,77940.0,77940.0,120.0,120.0
39602,Green-B,ADDED-1568817831,70113,290,78060.0,78060.0,120.0,120.0
39703,Green-B,ADDED-1568817831,70111,300,78180.0,78180.0,120.0,120.0


### Surrounding Trip Context for Missing Scheduled Travel Time

The five records with missing `scheduled_travel_time` were examined within their corresponding trip sequences. The output contains 124 rows across the affected trips, allowing the candidate records to be compared with other stops in the same trips.

The surrounding records generally contain valid scheduled travel times and calculated values, while the candidate rows contain the previously identified timestamp-derived values, including negative durations.

This provides additional context for determining whether the inconsistencies are isolated to the candidate stops or occur across a wider portion of the affected trips.

The next step is to narrow the output to the candidate stops and their immediate neighboring stops for closer comparison.

In [56]:
candidate_positions = scheduled_missing_candidates[
    ["trip_id", "stop_sequence"]
].copy()

candidate_positions["candidate_sequence"] = candidate_positions["stop_sequence"]

scheduled_local_context = scheduled_context.merge(
    candidate_positions[["trip_id", "candidate_sequence"]],
    on="trip_id",
    how="inner"
)

scheduled_local_context = scheduled_local_context[
    (
        scheduled_local_context["stop_sequence"]
        .sub(
            scheduled_local_context["candidate_sequence"]
        )
        .abs()
        <= 1
    )
].drop_duplicates()

scheduled_local_context[
    [
        "route_id",
        "trip_id",
        "stop_id",
        "stop_sequence",
        "scheduled_arrival_time",
        "scheduled_departure_time",
        "scheduled_travel_time",
        "calculated_scheduled_travel_time",
    ]
].sort_values(
    ["trip_id", "stop_sequence"]
)

,route_id,trip_id,stop_id,stop_sequence,scheduled_arrival_time,scheduled_departure_time,scheduled_travel_time,calculated_scheduled_travel_time
0,Green-C,ADDED-1568817376,70202,40,45780.0,45780.0,120.0,NaN
1,Green-D,ADDED-1568817376,70202,40,45660.0,45660.0,NaN,-120.0
22,Green-B,ADDED-1568817669,70196,50,66900.0,66900.0,NaN,540.0
47,Green-C,ADDED-1568817804,70197,60,76740.0,76740.0,180.0,NaN
48,Green-B,ADDED-1568817804,70197,60,76020.0,76020.0,NaN,-720.0
73,Green-C,ADDED-1568817806,70197,60,75180.0,75180.0,180.0,NaN
74,Green-B,ADDED-1568817806,70197,60,74760.0,74760.0,NaN,-420.0
99,Green-C,ADDED-1568817831,70197,60,76200.0,76200.0,180.0,NaN
100,Green-B,ADDED-1568817831,70197,60,75600.0,75600.0,NaN,-600.0


### Local Context of Missing Scheduled Travel Time Candidates

The five candidate records were compared with nearby stop sequences within their associated trips. The output shows that some `trip_id` values appear under more than one `route_id`, including the affected Green-C and Green-D records.

This indicates that `trip_id` alone does not uniquely identify the route context for these records. As a result, the surrounding records need to be examined using both `route_id` and `trip_id` to avoid combining records from different route contexts.

The timestamp inconsistencies therefore require a more specific local comparison before determining whether they are isolated to the candidate records.

In [57]:
candidate_keys = scheduled_missing_candidates[
    ["route_id", "trip_id", "stop_sequence"]
].copy()

local_context = scheduled_check.merge(
    candidate_keys,
    on=["route_id", "trip_id"],
    how="inner",
    suffixes=("", "_candidate")
)

local_context = local_context[
    (
        local_context["stop_sequence"]
        .sub(local_context["stop_sequence_candidate"])
        .abs()
        <= 1
    )
].copy()

local_context[
    [
        "route_id",
        "trip_id",
        "stop_id",
        "stop_sequence",
        "scheduled_arrival_time",
        "scheduled_departure_time",
        "scheduled_travel_time",
        "calculated_scheduled_travel_time",
    ]
].sort_values(
    ["route_id", "trip_id", "stop_sequence"]
)

,route_id,trip_id,stop_id,stop_sequence,scheduled_arrival_time,scheduled_departure_time,scheduled_travel_time,calculated_scheduled_travel_time
20,Green-B,ADDED-1568817669,70196,50,66900.0,66900.0,NaN,540.0
45,Green-B,ADDED-1568817804,70197,60,76020.0,76020.0,NaN,-720.0
70,Green-B,ADDED-1568817806,70197,60,74760.0,74760.0,NaN,-420.0
95,Green-B,ADDED-1568817831,70197,60,75600.0,75600.0,NaN,-600.0
0,Green-D,ADDED-1568817376,70202,40,45660.0,45660.0,NaN,-120.0


### Local Route and Trip Context for Missing Scheduled Travel Time

The five candidate records were re-examined using both `route_id` and `trip_id` to keep the route context consistent.

Only the five candidate records remain when restricting the comparison to the same route and trip and to adjacent `stop_sequence` values. No neighboring records are available within one stop sequence for these route/trip combinations.

This shows that the earlier trip-level context included records from different route contexts because some `trip_id` values occur under more than one `route_id`. The refined comparison avoids this mixing but does not provide adjacent stops for the five candidates.

The number and sequence range of records within each affected route/trip combination should therefore be examined before interpreting these timestamp inconsistencies further.

In [58]:
scheduled_route_trip_context = (
    scheduled_check[
        scheduled_check["trip_id"].isin(
            scheduled_missing_candidates["trip_id"]
        )
    ]
    .groupby(["route_id", "trip_id"])
    .agg(
        row_count=("stop_sequence", "size"),
        min_stop_sequence=("stop_sequence", "min"),
        max_stop_sequence=("stop_sequence", "max"),
        unique_stops=("stop_id", "nunique"),
    )
    .reset_index()
    .sort_values(["route_id", "trip_id"])
)

scheduled_route_trip_context

,route_id,trip_id,row_count,min_stop_sequence,max_stop_sequence,unique_stops
0,Green-B,ADDED-1568817669,24,50,310,24
1,Green-B,ADDED-1568817804,25,40,310,25
2,Green-B,ADDED-1568817806,25,40,310,25
3,Green-B,ADDED-1568817831,25,40,310,25
4,Green-C,ADDED-1568817376,1,40,40,1
5,Green-C,ADDED-1568817804,1,60,60,1
6,Green-C,ADDED-1568817806,1,60,60,1
7,Green-C,ADDED-1568817831,1,60,60,1
8,Green-D,ADDED-1568817376,20,40,570,20
9,Green-D,ADDED-1568817669,1,40,40,1


### Route and Trip Context of Missing Scheduled Travel Time Candidates

The route and trip combinations containing the five candidate records were examined to determine their available sequence ranges.

Four Green-B candidates belong to route/trip combinations containing 24–25 rows, while the Green-D candidate belongs to a combination containing 20 rows. The Green-C candidates each occur in route/trip combinations containing only one row.

The candidate records are also located at the minimum `stop_sequence` within their respective route/trip combinations. This explains why no adjacent rows were returned using the previous ±1 `stop_sequence` comparison.

The next step is to examine the actual previous and next rows within each route/trip sequence rather than relying on a numeric sequence difference.

In [59]:
scheduled_sequence_context = scheduled_check[
    scheduled_check["route_id"].isin(
        scheduled_missing_candidates["route_id"]
    )
    & scheduled_check["trip_id"].isin(
        scheduled_missing_candidates["trip_id"]
    )
].copy()

scheduled_sequence_context = (
    scheduled_sequence_context
    .sort_values(["route_id", "trip_id", "stop_sequence"])
)

scheduled_sequence_context["previous_stop_id"] = (
    scheduled_sequence_context
    .groupby(["route_id", "trip_id"])["stop_id"]
    .shift(1)
)

scheduled_sequence_context["previous_stop_sequence"] = (
    scheduled_sequence_context
    .groupby(["route_id", "trip_id"])["stop_sequence"]
    .shift(1)
)

scheduled_sequence_context["next_stop_id"] = (
    scheduled_sequence_context
    .groupby(["route_id", "trip_id"])["stop_id"]
    .shift(-1)
)

scheduled_sequence_context["next_stop_sequence"] = (
    scheduled_sequence_context
    .groupby(["route_id", "trip_id"])["stop_sequence"]
    .shift(-1)
)

scheduled_sequence_context[
    scheduled_sequence_context["scheduled_travel_time"].isna()
][
    [
        "route_id",
        "trip_id",
        "stop_id",
        "stop_sequence",
        "scheduled_arrival_time",
        "scheduled_departure_time",
        "scheduled_travel_time",
        "calculated_scheduled_travel_time",
        "previous_stop_id",
        "previous_stop_sequence",
        "next_stop_id",
        "next_stop_sequence",
    ]
]

,route_id,trip_id,stop_id,stop_sequence,scheduled_arrival_time,scheduled_departure_time,scheduled_travel_time,calculated_scheduled_travel_time,previous_stop_id,previous_stop_sequence,next_stop_id,next_stop_sequence
32682,Green-B,ADDED-1568817669,70196,50,66900.0,66900.0,NaN,540.0,NaN,NaN,70159,90.0
38493,Green-B,ADDED-1568817804,70202,40,NaN,NaN,NaN,NaN,NaN,NaN,70197,60.0
38598,Green-B,ADDED-1568817804,70197,60,76020.0,76020.0,NaN,-720.0,70202,40.0,70159,90.0
37736,Green-B,ADDED-1568817806,70202,40,NaN,NaN,NaN,NaN,NaN,NaN,70197,60.0
37881,Green-B,ADDED-1568817806,70197,60,74760.0,74760.0,NaN,-420.0,70202,40.0,70159,90.0
38182,Green-B,ADDED-1568817831,70202,40,NaN,NaN,NaN,NaN,NaN,NaN,70197,60.0
38379,Green-B,ADDED-1568817831,70197,60,75600.0,75600.0,NaN,-600.0,70202,40.0,70159,90.0
18095,Green-D,ADDED-1568817376,70202,40,45660.0,45660.0,NaN,-120.0,NaN,NaN,70198,70.0
32515,Green-D,ADDED-1568817669,70202,40,66360.0,66360.0,NaN,NaN,NaN,NaN,NaN,NaN


### Route and Trip Context for Missing Scheduled Travel Time

The missing `scheduled_travel_time` records were examined together with their previous and next stops within each `route_id` and `trip_id` combination.

Several candidate records occur at the beginning of their available route/trip sequence, while the Green-B candidates at `stop_sequence` 60 have previous stops whose scheduled timestamps are missing. As a result, the timestamp-derived values calculated earlier cannot be treated as valid same-route/trip reconstruction values for these records.

This also shows that `trip_id` alone is not sufficient for calculating the scheduled travel-time relationship because the same trip identifier can occur under different routes.

The scheduled travel-time relationship should therefore be revalidated using both `route_id` and `trip_id` before any further conclusions are made about reconstructing missing values.

In [60]:
scheduled_check_route_trip = df.sort_values(
    ["route_id", "trip_id", "stop_sequence"]
).copy()

scheduled_check_route_trip["previous_scheduled_departure"] = (
    scheduled_check_route_trip
    .groupby(["route_id", "trip_id"])["scheduled_departure_time"]
    .shift(1)
)

scheduled_check_route_trip["calculated_scheduled_travel_time"] = (
    scheduled_check_route_trip["scheduled_arrival_time"]
    - scheduled_check_route_trip["previous_scheduled_departure"]
)

scheduled_comparable_route_trip = scheduled_check_route_trip[
    scheduled_check_route_trip["scheduled_travel_time"].notna()
    & scheduled_check_route_trip["calculated_scheduled_travel_time"].notna()
].copy()

scheduled_comparable_route_trip[
    [
        "scheduled_travel_time",
        "calculated_scheduled_travel_time",
    ]
].describe()

,scheduled_travel_time,calculated_scheduled_travel_time
count,41688.000000,41688.000000
mean,132.143063,132.623777
std,55.440096,57.602155
min,0.000000,0.000000
25%,120.000000,120.000000
50%,120.000000,120.000000
75%,180.000000,180.000000
max,480.000000,1620.000000


### Corrected Scheduled Travel Time and Timestamp-Derived Values

The scheduled travel-time relationship was recalculated using both `route_id` and `trip_id` to preserve the route context.

There are 41,688 comparable rows. The mean `scheduled_travel_time` is 132.14 seconds compared with 132.62 seconds for the calculated value, while both have a median of 120 seconds and a 75th percentile of 180 seconds.

The corrected calculated values range from 0 to 1,620 seconds, compared with 0 to 480 seconds for the recorded `scheduled_travel_time`. The negative values seen in the earlier calculation are no longer present.

The relationship remains broadly consistent after correcting the grouping. The next step is to measure the exact match rate using the corrected calculation.

In [61]:
scheduled_comparable_route_trip["scheduled_travel_difference"] = (
    scheduled_comparable_route_trip["scheduled_travel_time"]
    - scheduled_comparable_route_trip["calculated_scheduled_travel_time"]
)

scheduled_comparable_route_trip[
    "scheduled_travel_difference"
].value_counts().head(10)

scheduled_travel_difference
 0.0      41610
-120.0       26
-180.0       19
-240.0        8
-300.0        6
-60.0         5
-660.0        3
-360.0        2
-420.0        2
-480.0        2
Name: count, dtype: int64

### Exact Match After Correcting the Route and Trip Grouping

The difference between `scheduled_travel_time` and `calculated_scheduled_travel_time` was recalculated using both `route_id` and `trip_id`.

There are 41,610 exact matches out of 41,688 comparable rows, leaving 78 rows with non-zero differences. The most frequent differences are -120 seconds (26 rows), -180 seconds (19 rows), and -240 seconds (8 rows).

This shows that the timestamp-derived value matches the recorded `scheduled_travel_time` for approximately 99.81% of comparable records after correcting the grouping.

The remaining mismatches are few but should be examined for their magnitude before the relationship is used to assess missing `scheduled_travel_time` values.

In [62]:
scheduled_mismatches_route_trip = scheduled_comparable_route_trip[
    scheduled_comparable_route_trip["scheduled_travel_difference"] != 0
].copy()

scheduled_mismatches_route_trip["absolute_difference"] = (
    scheduled_mismatches_route_trip["scheduled_travel_difference"].abs()
)

scheduled_mismatches_route_trip["absolute_difference"].describe()

count      78.000000
mean      256.923077
std       243.653510
min        60.000000
25%       120.000000
50%       180.000000
75%       285.000000
max      1440.000000
Name: absolute_difference, dtype: float64

### Magnitude of Corrected Scheduled Travel Time Mismatches

The magnitude of the 78 non-matching `scheduled_travel_difference` values was examined using their absolute differences.

The median absolute difference is 180 seconds, with the middle 50% of values ranging from 120 to 285 seconds. The mean is 256.92 seconds, while the minimum is 60 seconds and the maximum reaches 1,440 seconds.

The remaining mismatches are therefore meaningful rather than simple rounding differences, although they are less extreme than those identified using the earlier `trip_id`-only grouping.

The corrected timestamp relationship remains strong, but the remaining mismatches should be characterized further before assessing missing `scheduled_travel_time` values.

In [63]:
scheduled_mismatch_by_route_corrected = (
    scheduled_comparable_route_trip.assign(
        scheduled_travel_mismatch=(
            scheduled_comparable_route_trip["scheduled_travel_difference"] != 0
        )
    )
    .groupby("route_id")["scheduled_travel_mismatch"]
    .agg(["sum", "count"])
    .rename(
        columns={
            "sum": "mismatch_count",
            "count": "comparable_count",
        }
    )
)

scheduled_mismatch_by_route_corrected["mismatch_pct"] = (
    scheduled_mismatch_by_route_corrected["mismatch_count"]
    / scheduled_mismatch_by_route_corrected["comparable_count"]
    * 100
)

scheduled_mismatch_by_route_corrected.sort_values(
    "mismatch_pct", ascending=False
)

,mismatch_count,comparable_count,mismatch_pct
route_id,,,
Green-D,48,4992,0.961538
Green-C,12,5842,0.205409
Red,7,7111,0.098439
Green-B,5,7104,0.070383
Green-E,4,5686,0.070348
Blue,1,3901,0.025634
Orange,1,5676,0.017618
Mattapan,0,1376,0.000000


### Corrected Scheduled Travel Time Mismatch Rate by Route

The 78 corrected mismatches were compared across `route_id` to determine whether they remain concentrated on particular routes.

Green-D has the highest mismatch rate at 0.96%, with 48 mismatches among 4,992 comparable rows. Green-C has 12 mismatches (0.21%), followed by Red with 7 (0.10%). Green-B and Green-E have 5 and 4 mismatches respectively, while Blue and Orange each have 1 mismatch and Mattapan has none.

The remaining mismatches are therefore rare overall but are concentrated mainly on the Green routes, particularly Green-D. The next step is to examine the mismatch magnitude by route using the corrected grouping.

In [64]:
scheduled_mismatch_by_route_magnitude_corrected = (
    scheduled_mismatches_route_trip
    .groupby("route_id")["absolute_difference"]
    .agg(["count", "median", "mean", "max"])
    .rename(
        columns={
            "count": "mismatch_count",
            "median": "median_absolute_difference",
            "mean": "mean_absolute_difference",
            "max": "max_absolute_difference",
        }
    )
    .sort_values("median_absolute_difference", ascending=False)
)

scheduled_mismatch_by_route_magnitude_corrected

,mismatch_count,median_absolute_difference,mean_absolute_difference,max_absolute_difference
route_id,,,,
Green-B,5,180.0,288.0,900.0
Red,7,180.0,240.0,480.0
Green-D,48,180.0,280.0,1440.0
Green-E,4,150.0,150.0,180.0
Green-C,12,150.0,230.0,660.0
Blue,1,60.0,60.0,60.0
Orange,1,60.0,60.0,60.0


### Corrected Scheduled Travel Time Mismatch Magnitude by Route

The magnitude of the remaining `scheduled_travel_difference` values was compared across routes using the corrected `route_id` and `trip_id` grouping.

Green-B, Red, and Green-D each have a median absolute difference of 180 seconds. Green-B has a mean absolute difference of 288 seconds and a maximum of 900 seconds, while Green-D has the largest maximum at 1,440 seconds.

Green-E and Green-C have median absolute differences of 150 seconds, with maximum differences of 180 and 660 seconds respectively. Blue and Orange each have one mismatch with an absolute difference of 60 seconds.

Although Green-D has the highest mismatch frequency, its typical mismatch magnitude is not uniquely larger than the other routes. The next step is to examine the direction of the corrected mismatches.

In [65]:
scheduled_mismatch_direction_by_route_corrected = (
    scheduled_mismatches_route_trip
    .groupby("route_id")["scheduled_travel_difference"]
    .agg(["count", "median", "mean", "min", "max"])
    .rename(
        columns={
            "count": "mismatch_count",
            "median": "median_difference",
            "mean": "mean_difference",
            "min": "minimum_difference",
            "max": "maximum_difference",
        }
    )
    .sort_values("median_difference")
)

scheduled_mismatch_direction_by_route_corrected

,mismatch_count,median_difference,mean_difference,minimum_difference,maximum_difference
route_id,,,,,
Green-B,5,-180.0,-288.0,-900.0,-60.0
Green-D,48,-180.0,-280.0,-1440.0,-60.0
Red,7,-180.0,-240.0,-480.0,-120.0
Green-C,12,-150.0,-230.0,-660.0,-60.0
Green-E,4,-150.0,-150.0,-180.0,-120.0
Blue,1,-60.0,-60.0,-60.0,-60.0
Orange,1,-60.0,-60.0,-60.0,-60.0


### Direction of Corrected Scheduled Travel Time Mismatches

The direction of the remaining `scheduled_travel_difference` values was compared across routes.

All 78 mismatches are negative. Green-B, Green-D, and Red have median differences of -180 seconds, while Green-C and Green-E have median differences of -150 seconds. Blue and Orange each have a single mismatch of -60 seconds.

The mean differences are also negative across all affected routes, with Green-B having the largest mean difference at -288 seconds. The largest discrepancy occurs on Green-D at -1,440 seconds.

This indicates that the remaining mismatches are consistently directional rather than occurring in both directions. The next step is to apply the corrected `route_id` and `trip_id` relationship to the missing `scheduled_travel_time` records.

In [66]:
scheduled_missing_candidates_corrected = scheduled_check_route_trip[
    scheduled_check_route_trip["scheduled_travel_time"].isna()
    & scheduled_check_route_trip["calculated_scheduled_travel_time"].notna()
].copy()

scheduled_missing_candidates_corrected[
    "calculated_scheduled_travel_sign"
] = (
    scheduled_missing_candidates_corrected[
        "calculated_scheduled_travel_time"
    ]
    .apply(
        lambda x: "Positive" if x > 0
        else "Zero" if x == 0
        else "Negative"
    )
)

scheduled_missing_candidates_corrected[
    "calculated_scheduled_travel_sign"
].value_counts()

Series([], Name: count, dtype: int64)

### Missing Scheduled Travel Time After Correcting the Grouping

After recalculating the timestamp relationship using both `route_id` and `trip_id`, no missing `scheduled_travel_time` records have a non-missing `calculated_scheduled_travel_time` value.

This means the 5 candidate records identified under the earlier `trip_id`-only grouping were not valid reconstruction candidates. Their calculated values resulted from combining records across route contexts.

The corrected grouping therefore provides no timestamp-derived candidates for the missing `scheduled_travel_time` values. The next step is to examine which required timestamps are unavailable within the correct route/trip sequence.

In [67]:
scheduled_missing_timestamp_profile = (
    scheduled_check_route_trip.assign(
        scheduled_travel_missing=(
            scheduled_check_route_trip["scheduled_travel_time"].isna()
        ),
        previous_departure_missing=(
            scheduled_check_route_trip["previous_scheduled_departure"].isna()
        ),
        scheduled_arrival_missing=(
            scheduled_check_route_trip["scheduled_arrival_time"].isna()
        ),
    )
    .loc[
        lambda x: x["scheduled_travel_missing"]
    ]
    .groupby(
        [
            "previous_departure_missing",
            "scheduled_arrival_missing",
        ]
    )
    .size()
    .rename("row_count")
    .reset_index()
)

scheduled_missing_timestamp_profile

,previous_departure_missing,scheduled_arrival_missing,row_count
0,False,True,63
1,True,False,2041
2,True,True,154


### Scheduled Travel Time Missingness by Required Timestamp Availability

The missing `scheduled_travel_time` records were grouped by the availability of the current `scheduled_arrival_time` and the previous scheduled departure within the `route_id` and `trip_id` sequence.

The results show 63 rows with a missing `scheduled_arrival_time` but an available previous scheduled departure, 2,041 rows with an available scheduled arrival but no previous scheduled departure, and 154 rows where both are missing.

These patterns account for all 2,258 missing `scheduled_travel_time` values. Most missing values therefore occur because the previous scheduled departure is unavailable within the correct route and trip sequence.

This confirms that timestamp-based reconstruction is not currently supported for these records under the corrected grouping.

In [68]:
scheduled_missing_arrival_available = scheduled_check_route_trip[
    scheduled_check_route_trip["scheduled_travel_time"].isna()
    & scheduled_check_route_trip["scheduled_arrival_time"].notna()
].copy()

scheduled_missing_arrival_available["previous_stop_id"] = (
    scheduled_missing_arrival_available
    .groupby(["route_id", "trip_id"])["stop_id"]
    .shift(1)
)

scheduled_missing_arrival_available["previous_row_exists"] = (
    scheduled_missing_arrival_available["previous_stop_id"].notna()
)

scheduled_missing_arrival_available[
    "previous_row_exists"
].value_counts()

previous_row_exists
False    2041
Name: count, dtype: int64

### Previous Stop Availability for Missing Scheduled Travel Time

The 2,041 missing `scheduled_travel_time` records with an available `scheduled_arrival_time` were checked for the existence of a previous stop within the same `route_id` and `trip_id` sequence.

All **2,041 rows** have no previous row in their corresponding route/trip sequence. This means the missing previous scheduled departure is not caused by a missing value in an existing previous stop.

These records therefore represent the first available stop within their respective route/trip sequences, so a previous scheduled departure cannot be used to calculate `scheduled_travel_time`.

The remaining missing scheduled travel-time patterns should now be examined to determine whether any other structural relationship can explain the missing values.

In [69]:
scheduled_first_stop_missing = scheduled_missing_arrival_available[
    ~scheduled_missing_arrival_available["previous_row_exists"]
].copy()

scheduled_first_stop_missing[
    "stop_sequence"
].describe()

count    2041.000000
mean       25.756002
std        63.270802
min         1.000000
25%         1.000000
50%         1.000000
75%        40.000000
max       380.000000
Name: stop_sequence, dtype: float64

### Stop Sequence Position of Missing Scheduled Travel Time Records

The 2,041 records with an available `scheduled_arrival_time` but no previous row within the same `route_id` and `trip_id` sequence were examined by `stop_sequence`.

The sequence values range from 1 to 380, with a median of 1 and a 75th percentile of 40. This shows that the records are not all at `stop_sequence` 1, despite having no previous row in the available route/trip data.

These records therefore represent the first available row within their respective route/trip subsets, rather than necessarily the first stop of the full trip.

The next step is to examine the distribution of these records by `stop_sequence` to determine whether the missing values are concentrated at specific sequence positions.

In [70]:
scheduled_first_stop_missing["stop_sequence"].value_counts().sort_index()

stop_sequence
1      1358
20      140
40      154
50      298
60        4
310      85
380       2
Name: count, dtype: int64

### Distribution of Missing Scheduled Travel Time by Stop Sequence

The 2,041 records without a previous row were examined by `stop_sequence` to determine where the missing `scheduled_travel_time` values occur.

The largest group is at sequence 1 with 1,358 rows, followed by sequence 50 with 298 rows, sequence 40 with 154 rows, sequence 20 with 140 rows, sequence 310 with 85 rows, sequence 60 with 4 rows, and sequence 380 with 2 rows.

The missing values are therefore concentrated at a small number of sequence positions, with most occurring at sequence 1. However, a substantial number begin at later sequence positions.

The next step is to examine these sequence positions by `route_id` to determine whether the pattern is route-specific.

In [71]:
scheduled_first_stop_missing_by_route = (
    scheduled_first_stop_missing
    .groupby(["route_id", "stop_sequence"])
    .size()
    .rename("row_count")
    .reset_index()
    .sort_values(
        ["route_id", "stop_sequence"]
    )
)

scheduled_first_stop_missing_by_route

,route_id,stop_sequence,row_count
0,Blue,1,366
1,Blue,40,24
2,Green-B,50,166
3,Green-B,60,4
4,Green-C,1,2
5,Green-C,20,135
6,Green-D,1,1
7,Green-D,40,129
8,Green-D,310,85
9,Green-D,380,2


### Route Distribution of Missing Scheduled Travel Time by Stop Sequence

The 2,041 records without a previous row were examined by both `route_id` and `stop_sequence` to determine whether the sequence pattern is route-specific.

The missing records are distributed differently across routes. Blue has 390 records, Green-B 170, Green-C 137, Green-D 217, Green-E 343, Mattapan 87, Orange 283, and Red 453. Several routes also contain substantial missingness at later sequence positions, including Green-B at sequence 50, Green-C at sequence 20, and Green-D at sequences 40, 310, and 380.

This confirms that the missing `scheduled_travel_time` values are not limited to the first stop of the full trip. Instead, the available route/trip subsets begin at different points in the stop sequence.

The next step is to examine whether these missing records correspond to specific trip types or other structural characteristics.

In [72]:
scheduled_first_stop_missing["trip_type"] = (
    scheduled_first_stop_missing["trip_id"]
    .astype(str)
    .str.startswith("ADDED-")
    .map({True: "ADDED", False: "Regular"})
)

scheduled_first_stop_missing["trip_type"].value_counts()

trip_type
Regular    1553
ADDED       488
Name: count, dtype: int64

### Trip Type Distribution of Missing Scheduled Travel Time

The 2,041 records without a previous row were compared by trip type to determine whether the missing `scheduled_travel_time` values are concentrated among `ADDED-` trips.

There are 1,553 Regular trip records and 488 `ADDED-` trip records in this group.

The missingness therefore occurs in both trip types and is not limited to `ADDED-` trips. Regular trips account for the larger share of these records, so trip type alone does not explain the missing scheduled travel-time values.

The next step is to examine whether the missingness is associated with the availability of other scheduled timing fields.

In [73]:
scheduled_first_stop_missing[
    [
        "scheduled_arrival_time",
        "scheduled_departure_time",
    ]
].isna().sum()

scheduled_arrival_time      0
scheduled_departure_time    0
dtype: int64

### Scheduled Timestamp Availability for Missing Travel Time Records

The 2,041 records without a previous row were checked for missing current-stop scheduled timestamps.

Both `scheduled_arrival_time` and `scheduled_departure_time` are available for all 2,041 records. The missing `scheduled_travel_time` is therefore not caused by missing scheduled timestamps at the current stop.

Instead, the required previous stop is absent from the same `route_id` and `trip_id` sequence, so the scheduled travel-time interval cannot be calculated from the available records.

The next step is to examine the remaining 217 missing records where `scheduled_arrival_time` is also unavailable.

In [74]:
scheduled_missing_arrival = scheduled_check_route_trip[
    scheduled_check_route_trip["scheduled_travel_time"].isna()
    & scheduled_check_route_trip["scheduled_arrival_time"].isna()
].copy()

scheduled_missing_arrival[
    [
        "scheduled_arrival_time",
        "scheduled_departure_time",
        "previous_scheduled_departure",
    ]
].isna().sum()

scheduled_arrival_time          217
scheduled_departure_time        217
previous_scheduled_departure    154
dtype: int64

### Scheduled Timestamp Missingness in Remaining Travel Time Records

The 217 records with missing `scheduled_arrival_time` were checked for the availability of the current `scheduled_departure_time` and the previous scheduled departure.

Both `scheduled_arrival_time` and `scheduled_departure_time` are missing for all 217 records. A previous scheduled departure is also missing for 154 of these records.

This confirms that these records cannot be used for direct timestamp-based calculation of `scheduled_travel_time` because the current scheduled timestamps are unavailable.

The next step is to examine the 63 records where a previous scheduled departure exists despite both current scheduled timestamps being missing.